# Musanga V5 — NB08 native satellite detectability and production model

CPU, Google Drive and authorized Earth Engine access required. All targets are regenerated from the six NB07 V5 RGB mosaics. New UTM35N grids and satellite caches are stored exclusively under Final/V5/Outputs. No previous-version generated inputs. Primary 50m S2+embeddings, strict LOFO and 500m spatial-block CV (100m centre exclusion), annual/seasonal ablation, multi-resolution analysis, abundance sensitivity, optional independent forest-condition controls and balance-qualified spectra. Historical NICFI/context and UAV DSM-context experiments are not automatically regenerated; retained in V4 for provenance. The 1m UAV lattice approximates area, not exact native area; all models predict UAV teacher cover, not independent ground truth. The acquisition-year and preceding two-year pooled composites are fixed. Do not delete older folders until native execution and retention audit.

## 00 — Native manifest, source and grid configuration

In [ ]:
%pip -q install earthengine-api rasterio geopandas pyarrow scikit-learn scipy pandas matplotlib requests joblib
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime,timezone
from collections import defaultdict
import os,json,hashlib,time,math,requests
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin,Affine
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
V5_ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')) if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert V5_ROOT is not None
ROOT=V5_ROOT/'Outputs/08_satellite_reference';ROOT.mkdir(parents=True,exist_ok=True)
OUT=V5_ROOT/'Outputs/08_satellite_science_v5';OUT.mkdir(parents=True,exist_ok=True)
V4_OUT=OUT # compatibility variable for retained analysis functions; never a V4 path
NB07=V5_ROOT/'Outputs/07_RGB_unified_production_v5'
MANIFEST=NB07/'07_production_mosaic_manifest.json'
assert MANIFEST.is_file(),'NB07 must finish all six mosaics before NB08 execution.'
M=json.loads(MANIFEST.read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
SITES=('Baego','MOS','Yangambi','Yoko')
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
KEYS=['site','mosaic','row','col']
CRS='EPSG:32635';TILE_M=1500;LATTICE_M=1;MIN_SUPPORT=.90
SCALES=[10,25,30,50,75,100]
EE_PROJECT='musanga-511110' # change to an authorized project when transferring
S2_SOURCE='COPERNICUS/S2_SR_HARMONIZED'
EMB_SOURCE='GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
SCL_REJECT=[0,1,3,7,8,9,10,11]
INDEX_NAMES=['NDVI','NDMI','NBR','NDRE']
EMB_BANDS=[f'A{i:02d}' for i in range(64)]
ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
SEED=20261120;SEED17=SEED;CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
RUN_STRUCTURE=True  # supplementary; independent GEE assets may require access
RUN_ABUNDANCE_SENSITIVITY=True
def digest(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,obj):
 p=Path(p);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(d,p):
 p=Path(p);tmp=p.with_suffix('.partial');d.to_parquet(tmp,index=False);os.replace(tmp,p)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()
# CPU-only recovery of a missing final mosaic from its verified native V5 stripes.
# Never use a legacy raster, never rerun GPU inference, never overwrite an existing final.
import shutil
from rasterio.windows import Window
def recover_missing_mosaic(item,p):
 if p.is_file():return
 key=item['digest'];w=int(item['width']);h=int(item['height'])
 tr=Affine(*item['transform'][:6]);crs=rasterio.crs.CRS.from_string(item['crs'])
 stripe_h=1024
 folder=NB07/'stripe_cache'/item['name']/key[:20]
 expected=math.ceil(h/stripe_h)
 assert folder.is_dir(),f'Missing final mosaic AND stripe cache: {p}'
 stripes=[];total=0;tiles=0
 for i in range(expected):
  a=folder/f'{i:05d}.tif';meta=a.with_suffix('.json')
  assert a.is_file() and meta.is_file(),f'Missing stripe {i}: {folder}'
  rec=json.loads(meta.read_text())
  assert rec['signature']==key and rec['stripe']==i
  with rasterio.open(a) as ds:
   assert ds.tags().get('signature')==key and ds.tags().get('model_sha')==M['model_sha'] and ds.tags().get('policy_sha')==M['policy_sha']
   assert ds.count==2 and ds.dtypes==('float32','float32') and ds.nodata==-9999 and ds.crs==crs
   assert ds.width==w and ds.height==min(stripe_h,h-i*stripe_h)
   assert ds.transform.almost_equals(tr*Affine.translation(0,i*stripe_h))
  total+=int(rec['valid_pixels']);tiles+=int(rec['tiles']);stripes.append((a,rec))
 assert total==item['valid_pixels'] and tiles==item['count']
 print(f'RECOVER {item["name"]}: {expected} verified stripe headers; CPU assembly only',flush=True)
 local=Path('/content')/f'{item["name"]}_{key[:12]}_recovery.tif'
 profile=dict(driver='GTiff',width=w,height=h,count=2,dtype='float32',nodata=-9999,crs=crs,transform=tr,tiled=True,blockxsize=256,blockysize=256,compress='DEFLATE',predictor=3,BIGTIFF='YES',SPARSE_OK='TRUE')
 with rasterio.open(local,'w',**profile) as dst:
  dst.set_band_description(1,'Musanga probability');dst.set_band_description(2,'Musanga binary at 0.5')
  dst.update_tags(signature=key,model_sha=M['model_sha'],policy_sha=M['policy_sha'])
  for i,(a,rec) in enumerate(stripes):
   count=0
   with rasterio.open(a) as src:
    for _,win in src.block_windows(1):
     z=src.read(window=win);v=z[0]!=-9999
     assert np.array_equal(v,z[1]!=-9999)
     assert np.isfinite(z).all() and np.all((z[0][v]>=0)&(z[0][v]<=1))
     assert np.array_equal(z[1][v],(z[0][v]>=.5).astype(np.float32))
     count+=int(v.sum())
     if v.any():dst.write(z,window=Window(win.col_off,win.row_off+i*stripe_h,win.width,win.height))
   assert count==rec['valid_pixels'],(a,count,rec['valid_pixels'])
   print(f'Recovery {item["name"]}: stripe {i+1}/{expected}',flush=True)
 # Copy to final name directly (avoid the previous Drive rename failure).
 # A completion receipt is written only after read-back byte hash verification.
 assert not p.exists(),f'Final appeared during recovery: {p}; stop concurrent writers'
 def sha_file(q):
  h=hashlib.sha256()
  with open(q,'rb') as f:
   for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
  return h.hexdigest()
 digest_local=sha_file(local)
 with open(local,'rb') as f,open(p,'xb') as out:
  shutil.copyfileobj(f,out,8*1024*1024);out.flush();os.fsync(out.fileno())
 assert p.stat().st_size==local.stat().st_size and sha_file(p)==digest_local
 with rasterio.open(p) as ds:
  assert ds.width==w and ds.height==h and ds.tags().get('signature')==key
 atomic_json(p.with_suffix('.recovery.json'),dict(signature=key,sha256=digest_local,valid_pixels=total,stripes=expected))
 local.unlink()
 print('RECOVERED AND VERIFIED',p,flush=True)

SOURCE_META=[];MOSAICS=defaultdict(list)
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 recover_missing_mosaic(item,p)
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest']
  assert ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5'),ds.descriptions
  bounds=list(ds.bounds)
 MOSAICS[item['site']].append((item['name'],p,bounds))
 SOURCE_META.append(dict(name=item['name'],site=item['site'],digest=item['digest'],size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns,bounds=bounds))
assert set(MOSAICS)==set(SITES)
for k in MOSAICS:MOSAICS[k].sort(key=lambda t:t[0])
TARGET_CFG=dict(version='v5_native_grid_v1',model=M['model_sha'],policy=M['policy_sha'],mosaics=SOURCE_META,crs=CRS,origin=[0,0],tile_m=TILE_M,lattice_m=LATTICE_M,scales=SCALES,support=MIN_SUPPORT,method='1m nearest valid support and binary point sampling; best-supported mosaic per cell; no blending')
TARGET_ID=digest(TARGET_CFG)[:20];TARGET_DIR=ROOT/'uav_targets'/TARGET_ID;TARGET_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(TARGET_DIR/'config.json',TARGET_CFG)
# Geometry is only used for candidate tile pruning; the map's actual NoData mask is authoritative.
raw=json.loads((V5_ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
roi_path=Path(raw['roi'])
if not roi_path.is_file():roi_path=V5_ROOT.parent/'Data/Ortho_RGB'/roi_path.name
roi=unary_union(gpd.read_file(roi_path).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
TILES={}
for forest in SITES:
 keys=set()
 for name,p,b in MOSAICS[forest]:
  geom=roi.intersection(box(*b));prepared=prep(geom)
  if geom.is_empty:continue
  xmin,ymin,xmax,ymax=geom.bounds
  for tx in range(math.floor(xmin/TILE_M),math.ceil(xmax/TILE_M)):
   for ty in range(math.floor(ymin/TILE_M),math.ceil(ymax/TILE_M)):
    if prepared.intersects(box(tx*TILE_M,ty*TILE_M,(tx+1)*TILE_M,(ty+1)*TILE_M)):keys.add((tx,ty))
 TILES[forest]=sorted(keys)
 assert keys,forest
print('Candidate tiles per forest:',{k:len(v) for k,v in TILES.items()})
print('Native V5 target fingerprint:',TARGET_ID)


## 01 — Generate resumable UAV targets at six resolutions

In [ ]:
# One bounded 1m lattice per tile and mosaic; all six scales derived from that lattice.
# A nearest-neighbour lattice is an area approximation, not exact native-pixel area.
def tile_bounds(tx,ty):return (tx*TILE_M,ty*TILE_M,(tx+1)*TILE_M,(ty+1)*TILE_M)
def tile_transform(tx,ty,spacing):
 x0,y0,x1,y1=tile_bounds(tx,ty);return from_origin(x0,y1,spacing,spacing)
def reduce_grid(a,scale,spacing):
 n=int(scale/spacing);h,w=a.shape;assert h%n==0 and w%n==0
 return a.reshape(h//n,n,w//n,n).sum(axis=(1,3))
TARGET_COLS=['site','mosaic','scale_m','row','col','x','y','target_fraction','mean_probability','uav_valid_fraction','tile_x','tile_y']
def target_tile(forest,tx,ty,opened):
 dest=TARGET_DIR/f'{forest}_{tx}_{ty}.parquet'
 if dest.is_file():
  d=pd.read_parquet(dest);assert list(d.columns)==TARGET_COLS
  return d
 b=tile_bounds(tx,ty);tr=tile_transform(tx,ty,LATTICE_M);n=TILE_M//LATTICE_M
 best={}
 for scale in SCALES:
  k=TILE_M//scale;best[scale]=(np.zeros((k,k),np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),-1,np.int16))
 for j,(name,p,bounds) in enumerate(MOSAICS[forest]):
  if not box(*b).intersects(box(*bounds)):continue
  ds=opened[name]
  with WarpedVRT(ds,crs=CRS,transform=tr,width=n,height=n,resampling=Resampling.nearest,nodata=-9999.) as vrt:
   z=vrt.read([1,2],masked=True)
  a=z.filled(-9999.).astype(np.float32,copy=False)
  valid=(~np.ma.getmaskarray(z).any(axis=0))&np.isfinite(a).all(axis=0)&(a[0]>=0)&(a[0]<=1)&((a[1]==0)|(a[1]==1))
  assert np.array_equal(a[1][valid],(a[0][valid]>=.5).astype(np.float32))
  if not valid.any():continue
  for scale in SCALES:
   count=reduce_grid(valid.astype(np.int32),scale,LATTICE_M)
   support=count/float((scale/LATTICE_M)**2)
   total=reduce_grid(np.where(valid,a[1],0.),scale,LATTICE_M)
   prob=reduce_grid(np.where(valid,a[0],0.),scale,LATTICE_M)
   v,c,pb,idx=best[scale];take=(support>=MIN_SUPPORT)&(support>v)
   v[take]=support[take];c[take]=total[take]/count[take];pb[take]=prob[take]/count[take];idx[take]=j
 frames=[]
 for scale,(v,c,pb,idx) in best.items():
  rr,cc=np.where(idx>=0)
  if not len(rr):continue
  x=b[0]+(cc+.5)*scale;y=b[3]-(rr+.5)*scale
  frames.append(pd.DataFrame(dict(site=forest,mosaic=[MOSAICS[forest][j][0] for j in idx[rr,cc]],scale_m=scale,
   row=np.floor(-y/scale).astype('int64'),col=np.floor(x/scale).astype('int64'),x=x,y=y,target_fraction=c[rr,cc],mean_probability=pb[rr,cc],uav_valid_fraction=v[rr,cc],tile_x=tx,tile_y=ty)))
 d=pd.concat(frames,ignore_index=True)[TARGET_COLS] if frames else pd.DataFrame(columns=TARGET_COLS)
 assert not d.duplicated(['site','scale_m','row','col']).any()
 atomic_parquet(d,dest)
 return d
target_parts=[];t0=time.monotonic()
for forest in SITES:
 with ExitStack() as stack:
  opened={name:stack.enter_context(rasterio.open(p)) for name,p,b in MOSAICS[forest]}
  for i,(tx,ty) in enumerate(TILES[forest],1):
   d=target_tile(forest,tx,ty,opened)
   if len(d):target_parts.append(d)
   print(f'UAV {forest} {i}/{len(TILES[forest])}: {len(d):,} cells across scales | {(time.monotonic()-t0)/60:.1f} min',flush=True)
TARGET=pd.concat(target_parts,ignore_index=True)
del target_parts
assert not TARGET.duplicated(['site','scale_m','row','col']).any()
assert TARGET.target_fraction.between(0,1).all() and TARGET.uav_valid_fraction.ge(MIN_SUPPORT-1e-6).all()
assert TARGET.groupby('scale_m').site.nunique().eq(4).all()
atomic_parquet(TARGET,TARGET_DIR/'all_scales_uav_reference.parquet')
display(TARGET.groupby(['scale_m','site']).target_fraction.agg(n='size',mean='mean',max='max').round(3))


### 01B — Native-area validation of the 1m sampling approximation
Up to 24 fixed 50m windows; exact native pixel overlap weights. This is a spot check, not validation of every cell.

In [ ]:
# Audit the approximation against exact native-pixel overlap area on a fixed sample.
# This reads at most 24 individual 50m windows, not entire mosaics.
from rasterio.windows import from_bounds,Window
AREA_QA_PATH=TARGET_DIR/'01_native_area_spotcheck_50m.csv'
samples=[]
for forest,g in TARGET[TARGET.scale_m.eq(50)].groupby('site'):
 boundary=g[g.uav_valid_fraction.lt(.98)]
 interior=g[g.uav_valid_fraction.ge(.98)]
 for label,z in [('boundary',boundary),('interior',interior)]:
  if len(z):
   q=z.sample(n=min(3,len(z)),random_state=SEED).copy();q['sample_type']=label;samples.append(q)
sample=pd.concat(samples,ignore_index=True)
if not AREA_QA_PATH.is_file():
 qa=[]
 for (forest,mosaic),g in sample.groupby(['site','mosaic']):
  p=next(p for name,p,b in MOSAICS[forest] if name==mosaic)
  with rasterio.open(p) as ds:
   tf=ds.transform;assert tf.a>0 and tf.e<0 and abs(tf.b)<1e-9 and abs(tf.d)<1e-9
   for row in g.itertuples():
    xmin=row.x-25;xmax=row.x+25;ymin=row.y-25;ymax=row.y+25
    raw=from_bounds(xmin,ymin,xmax,ymax,tf)
    c0=max(0,math.floor(raw.col_off));r0=max(0,math.floor(raw.row_off))
    c1=min(ds.width,math.ceil(raw.col_off+raw.width));r1=min(ds.height,math.ceil(raw.row_off+raw.height))
    assert c1>c0 and r1>r0
    a=ds.read(2,window=Window(c0,r0,c1-c0,r1-r0),masked=True)
    v=(~np.ma.getmaskarray(a))&np.isin(a.filled(-9999),[0,1])
    left=tf.c+np.arange(c0,c1)*tf.a
    top=tf.f+np.arange(r0,r1)*tf.e
    dx=np.maximum(0,np.minimum(left+tf.a,xmax)-np.maximum(left,xmin))
    dy=np.maximum(0,np.minimum(top,ymax)-np.maximum(top+tf.e,ymin))
    weights=dy[:,None]*dx[None,:]
    area=float(np.sum(weights*v));cover=float(np.sum(weights*np.where(v,a.filled(0),0))/area) if area>0 else np.nan
    qa.append(dict(site=forest,mosaic=mosaic,row=row.row,col=row.col,sample_type=row.sample_type,
      lattice_cover=row.target_fraction,native_cover=cover,lattice_support=row.uav_valid_fraction,native_support=area/2500,
      cover_delta_pp=100*(row.target_fraction-cover),support_delta_pp=100*(row.uav_valid_fraction-area/2500)))
 QAREA=pd.DataFrame(qa);tmp=AREA_QA_PATH.with_suffix('.partial');QAREA.to_csv(tmp,index=False);os.replace(tmp,AREA_QA_PATH)
else:QAREA=pd.read_csv(AREA_QA_PATH)
assert len(QAREA)==len(sample)
display(QAREA.round(4))
print('Native area spot-check | cover MAE (pp):',QAREA.cover_delta_pp.abs().mean(),'| max abs support error (pp):',QAREA.support_delta_pp.abs().max())
if QAREA.cover_delta_pp.abs().max()>2 or QAREA.support_delta_pp.abs().max()>2:
 print('WARNING: >2 pp spot-check difference; inspect lattice accuracy before publication.')


## 02 — Native acquisition-year Sentinel-2, 3-year pooled quarters and annual embeddings

In [ ]:
# Direct snapped 10 m Earth Engine downloads; no old predictor files.
import ee
try:ee.Initialize(project=EE_PROJECT)
except Exception:
 ee.Authenticate()
 ee.Initialize(project=EE_PROJECT)
SAT_CFG=dict(version='v5_ee_10m_v1',crs=CRS,tile_m=TILE_M,years=YEARS,s2=S2_SOURCE,embedding=EMB_SOURCE,mask=SCL_REJECT,annual_edge_mask=['B8A','B9'],annual_20m_resampling='bilinear',annual_indices=INDEX_NAMES,quarter_mask='SCL only, original NB04A',quarter_years='year-2 through year',reflectance_order=S2_BANDS,grid_origin=[0,0],scaling=.0001,brdf='none')
SAT_ID=digest(SAT_CFG)[:20];SAT_DIR=ROOT/'satellite_10m'/SAT_ID;SAT_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(SAT_DIR/'config.json',SAT_CFG)
def ee_region(tx,ty):
 b=transform_bounds(CRS,'EPSG:4326',*tile_bounds(tx,ty),densify_pts=21)
 return ee.Geometry.Rectangle(list(b),proj='EPSG:4326',geodesic=False)
def clean_annual(im):
 valid=im.select('SCL').remap(SCL_REJECT,[0]*len(SCL_REJECT),1).eq(1)
 valid=valid.And(im.select('B8A').mask()).And(im.select('B9').mask())
 return ee.Image.cat([im.select(S2_BANDS[:4]),im.select(S2_BANDS[4:]).resample('bilinear')]).updateMask(valid).copyProperties(im,['system:time_start'])
def clean_quarter(im):
 valid=im.select('SCL').remap(SCL_REJECT,[0]*len(SCL_REJECT),1).eq(1)
 return im.select(S2_BANDS).updateMask(valid).multiply(.0001).copyProperties(im,['system:time_start'])
def indices(im):
 return ee.Image.cat([im,im.normalizedDifference(['B8','B4']).rename('NDVI'),im.normalizedDifference(['B8','B11']).rename('NDMI'),im.normalizedDifference(['B8','B12']).rename('NBR'),im.normalizedDifference(['B8A','B5']).rename('NDRE')])
def sat_image(forest,tx,ty,family):
 year=YEARS[forest];area=ee_region(tx,ty)
 if family=='EMB':
  coll=ee.ImageCollection(EMB_SOURCE).filterBounds(area).filterDate(f'{year}-01-01',f'{year+1}-01-01').sort('system:index')
  image=coll.mosaic().select(EMB_BANDS)
  bands=EMB_BANDS
 elif family=='ANNUAL':
  coll=ee.ImageCollection(S2_SOURCE).filterBounds(area).filterDate(f'{year}-01-01',f'{year+1}-01-01').map(clean_annual)
  image=ee.Image.cat([indices(coll.median().multiply(.0001)),coll.select('B2').count().unmask(0).rename('N_CLEAR')])
  bands=S2_BANDS+INDEX_NAMES+['N_CLEAR']
 else:
  q=int(family[1:]);coll=ee.ImageCollection(S2_SOURCE).filterBounds(area).filterDate(f'{year-2}-01-01',f'{year+1}-01-01').filter(ee.Filter.calendarRange(q*3-2,q*3,'month')).map(clean_quarter)
  image=ee.Image.cat([coll.median().select(S2_BANDS),coll.select('B2').count().unmask(0).rename('N_CLEAR')])
  bands=S2_BANDS+['N_CLEAR']
 return image.select(bands).unmask(-9999,sameFootprint=False).toFloat(),bands,coll
def fetch_image(image,bands,folder,forest,tx,ty,family,recipe_id,collection=None):
 folder.mkdir(parents=True,exist_ok=True)
 path=folder/f'{forest}_{tx}_{ty}_{family}.tif';meta=path.with_suffix('.json')
 tf=tile_transform(tx,ty,10);n=TILE_M//10
 def verify(p):
  with rasterio.open(p) as ds:
   assert ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.shape==(n,n) and ds.count==len(bands)
   assert ds.transform.almost_equals(tf,precision=1e-6) and ds.nodata==-9999
   assert list(ds.descriptions)==bands and ds.tags().get('recipe')==recipe_id
   a=ds.read();assert np.isfinite(a).all()
   return a
 if path.is_file() and meta.is_file():
  assert json.loads(meta.read_text())['recipe']==recipe_id
  return verify(path)
 # Preserve exact catalogue image IDs in the V5 cache provenance.
 ids=collection.aggregate_array('system:index').getInfo() if collection is not None else None
 if collection is not None:assert len(ids)>0,(forest,tx,ty,family,'no scenes')
 params=dict(crs=CRS,crs_transform=list(tf)[:6],dimensions=[n,n],format='GEO_TIFF',filePerBand=False)
 error=None
 for attempt in range(4):
  try:
   url=image.getDownloadURL(params)
   response=requests.get(url,timeout=(30,300));response.raise_for_status()
   with MemoryFile(response.content) as mem,mem.open() as ds:
    assert ds.count==len(bands) and ds.shape==(n,n) and ds.crs==rasterio.crs.CRS.from_string(CRS),(family,ds.shape,ds.crs)
    assert ds.transform.almost_equals(tf,precision=1e-6),(family,ds.transform,tf)
    a=ds.read().astype(np.float32)
   assert np.isfinite(a).all()
   tmp=path.with_suffix('.partial.tif')
   with rasterio.open(tmp,'w',driver='GTiff',width=n,height=n,count=len(bands),dtype='float32',crs=CRS,transform=tf,nodata=-9999.,compress='deflate',tiled=True) as dst:
    dst.write(a)
    for j,b in enumerate(bands,1):dst.set_band_description(j,b)
    dst.update_tags(recipe=recipe_id,site=forest,family=family)
   os.replace(tmp,path);atomic_json(meta,dict(recipe=recipe_id,source_image_ids=ids,retrieved_utc=datetime.now(timezone.utc).isoformat(),bands=bands))
   return verify(path)
  except Exception as exc:
   error=exc;print('EE retry',forest,family,tx,ty,attempt+1,str(exc)[:220],flush=True);time.sleep(2**attempt)
 raise RuntimeError(f'EE download failed for {forest} {family} {tx} {ty}; completed caches are preserved') from error
def get_sat(forest,tx,ty,family):
 img,bands,coll=sat_image(forest,tx,ty,family)
 return fetch_image(img,bands,SAT_DIR,forest,tx,ty,family,SAT_ID,coll)
def aggregate_features(a,scale,count_indices=()):
 # Expand 10m pixels into equal 5m quadrants for exact aligned area weights
 # at 25/75m as well as integer multiples of 10m. No artificial new information.
 good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)
 for i in count_indices:good &= a[i]>0
 base=np.repeat(np.repeat(good,2,axis=0),2,axis=1)
 n=scale//5;den=reduce_grid(base.astype(np.int32),scale,5)
 vals=[]
 for b in a:
  z=np.repeat(np.repeat(np.where(good,b,0.),2,axis=0),2,axis=1)
  sums=reduce_grid(z,scale,5)
  vals.append(np.divide(sums,den,out=np.full(sums.shape,np.nan,dtype=np.float32),where=den>0))
 return np.stack(vals),den/float(n*n)
JOIN_CFG=dict(version='v5_native_join_v1',target=TARGET_ID,satellite=SAT_ID,support=MIN_SUPPORT,feature_policy='90% jointly valid 10m samples within each family; common valid cells across families for 50m')
JOIN_ID=digest(JOIN_CFG)[:20];JOIN_DIR=ROOT/'joined_tiles'/JOIN_ID;JOIN_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(JOIN_DIR/'config.json',JOIN_CFG)
def extract_join(forest,tx,ty,d):
 annual=get_sat(forest,tx,ty,'ANNUAL')
 extra=[]
 if d.scale_m.eq(50).any():
  extra=[(get_sat(forest,tx,ty,f'Q{q}'),[10],QUARTER[(q-1)*11:q*11]) for q in range(1,5)]
  extra.append((get_sat(forest,tx,ty,'EMB'),[],EMB))
 pieces=[]
 for scale,g in d.groupby('scale_m'):
  b=tile_bounds(tx,ty);rr=np.floor((b[3]-g.y.to_numpy())/scale).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/scale).astype(int)
  assert np.all(rr>=0) and np.all(cc>=0) and np.all(rr<TILE_M//scale) and np.all(cc<TILE_M//scale)
  vals,cov=aggregate_features(annual,int(scale),[14])
  h=g.copy().reset_index(drop=True)
  for i,c in enumerate(ANNUAL):h[c]=vals[i,rr,cc]
  h['annual_support']=cov[rr,cc];h=h[h.annual_support.ge(MIN_SUPPORT-1e-6)].copy()
  h['all_features_valid']=False
  if scale==50 and len(h):
   rr=np.floor((b[3]-h.y.to_numpy())/scale).astype(int);cc=np.floor((h.x.to_numpy()-b[0])/scale).astype(int)
   support=np.ones(len(h),bool)
   for a,counts,names in extra:
    v,cov=aggregate_features(a,50,counts)
    for i,c in enumerate(names):h[c]=v[i,rr,cc]
    support &= cov[rr,cc]>=MIN_SUPPORT-1e-6
   h['all_features_valid']=support
  pieces.append(h)
 return pd.concat(pieces,ignore_index=True)
# Keep wide 119-feature rows only at 50m; narrow tables at all other scales.
# Avoid concatenating millions of finer-scale rows with unused quarterly/embedding columns.
parts_by_scale=defaultdict(list)
groups=TARGET.groupby(['site','tile_x','tile_y'],sort=True);total=groups.ngroups
for i,((forest,tx,ty),d) in enumerate(groups,1):
 tx=int(tx);ty=int(ty);p=JOIN_DIR/f'{forest}_{tx}_{ty}.parquet'
 if p.is_file():
  z=pd.read_parquet(p);print('JOIN CACHE',forest,tx,ty,flush=True)
 else:
  z=extract_join(forest,tx,ty,d);atomic_parquet(z,p)
  print('JOIN SAVED',forest,tx,ty,len(z),flush=True)
 for k,q in z.groupby('scale_m'):
  cols=TARGET_COLS+ANNUAL+['annual_support']
  if k==50:cols+=QUARTER+EMB+['all_features_valid']
  # Some tiles lack 50m valid coverage; those never enter this branch.
  parts_by_scale[int(k)].append(q.reindex(columns=cols).copy())
 del z
 print(f'{i}/{total} satellite tiles complete',flush=True)
del groups
J50=pd.concat(parts_by_scale[50],ignore_index=True)
assert not J50.duplicated(['site','row','col']).any()
D=J50[J50.all_features_valid.eq(True)].copy()
D=D[np.isfinite(D[ALL_FEATURES+Q_COUNTS].to_numpy(float)).all(axis=1)&(D[Q_COUNTS].to_numpy(float)>=1).all(axis=1)].sort_values(KEYS).reset_index(drop=True)
assert set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any() and D.target_fraction.between(0,1).all()
D4=D.copy() # compatibility alias for retained scientific analysis code
SOURCE_SHA=dataframe_sha(D[KEYS+['x','y','target_fraction']+ALL_FEATURES])
CONFIG=dict(version='NB08_V5_native_50m',target_id=TARGET_ID,satellite_id=SAT_ID,join_id=JOIN_ID,source_sha256=SOURCE_SHA,n=len(D),features=FEATURE_SETS,seed=SEED,cap=CAP_PER_TRAIN_FOREST,model=MODEL,thresholds=THRESHOLDS)
SIGNATURE=digest(CONFIG)[:20];RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
atomic_json(RUN/'config.json',CONFIG)
REF=ROOT/f'native_50m_reference_{SIGNATURE}.parquet';atomic_parquet(D,REF)
scale_hashes={}
scale_cols=['site','scale_m','row','col','x','y','target_fraction','uav_valid_fraction']+ANNUAL[:14]
for k in SCALES:
 q=pd.concat([a[scale_cols] for a in parts_by_scale[k]],ignore_index=True).sort_values(['site','row','col']).reset_index(drop=True)
 assert not q.duplicated(['site','row','col']).any()
 assert set(q.site)==set(SITES) and np.isfinite(q[ANNUAL[:14]].to_numpy(float)).all()
 scale_hashes[str(k)]=dataframe_sha(q)
 del q
SCALE_CFG=dict(version='v5_annual_scale_v1',join=JOIN_ID,table_sha=scale_hashes,scales=SCALES)
SCALE_RUN=OUT/'native_scale_LOFO'/digest(SCALE_CFG)[:20];SCALE_RUN.mkdir(parents=True,exist_ok=True)
atomic_json(SCALE_RUN/'config.json',SCALE_CFG)
for k in SCALES:
 q=pd.concat([a[scale_cols] for a in parts_by_scale.pop(k)],ignore_index=True)
 q=q.rename(columns={c:f'S2_{i:02d}' for i,c in enumerate(ANNUAL[:14],1)})
 for forest,part in q.groupby('site'):
  part=part.sort_values(['row','col']).reset_index(drop=True)
  atomic_parquet(part,SCALE_RUN/f'{forest}_{k}m.parquet')
 del q
del parts_by_scale,J50
assert all((SCALE_RUN/f'{s}_{k}m.parquet').is_file() for s in SITES for k in SCALES)
display(D.groupby('site').target_fraction.agg(n='size',mean='mean',std='std',max='max').round(3))
print('Complete native V5 reference:',REF)


## 03 — Strict leave-one-forest-out (50m common predictor support)

In [ ]:
y=D.target_fraction.to_numpy(np.float32)
site=D.site.to_numpy()
def balanced_draw(pool,heldout):
 rng=np.random.default_rng(SEED+sum(map(ord,heldout)))
 chosen=[]
 for forest in SITES:
  ix=pool[site[pool]==forest]
  if not len(ix):continue
  cap=min(len(ix),CAP_PER_TRAIN_FOREST)
  if len(ix)<=cap:
   chosen.extend(ix.tolist());continue
  strata=np.digitize(y[ix],[.10,.20,.40])
  bins={j:ix[strata==j] for j in range(4)}
  pick=[]
  # Keep up to a quarter of training sample above 40% where available.
  take=min(len(bins[3]),int(np.ceil(.25*cap)))
  if take:
   ids=rng.choice(bins[3],take,replace=False)
   pick.extend(ids.tolist());bins[3]=np.setdiff1d(bins[3],ids)
  while len(pick)<cap:
   active=[k for k in bins if len(bins[k])]
   assert active
   for k in active:
    n=min(len(bins[k]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
    if n>0:
     ids=rng.choice(bins[k],n,replace=False)
     pick.extend(ids.tolist());bins[k]=np.setdiff1d(bins[k],ids)
    if len(pick)>=cap:break
  chosen.extend(pick)
 return np.asarray(sorted(chosen),dtype=int)
predictions=[]
audit=[]
for held in SITES:
 test=np.flatnonzero(site==held)
 train_pool=np.flatnonzero(site!=held)
 train=balanced_draw(train_pool,held)
 assert not np.intersect1d(train,test).size
 audit.append(dict(heldout=held,train_cells=len(train),test_cells=len(test),
                   high_train=int((y[train]>=.40).sum())))
 # The constant null is learned from the **same** training draw.
 null_value=float(np.mean(y[train]))
 frame=D.iloc[test][KEYS+['x','y','target_fraction']].copy().reset_index(drop=True)
 frame['heldout']=held
 frame['M0_mean']=null_value
 for model_name,cols in FEATURE_SETS.items():
  cache=RUN/f'{held}_{model_name}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test_index'],test)
    assert z['source_sha256'].item()==SOURCE_SHA
    pred=z['pred'].astype(np.float32)
   print('CACHE',held,model_name,flush=True)
  else:
   t=time.monotonic()
   model=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   model.fit(D.iloc[train][cols].to_numpy('float32'),y[train])
   pred=np.clip(model.predict(D.iloc[test][cols].to_numpy('float32')),0,1).astype(np.float32)
   tmp=cache.with_suffix('.partial')
   with open(tmp,'wb') as f:
    np.savez_compressed(f,test_index=test,pred=pred,source_sha256=SOURCE_SHA)
   os.replace(tmp,cache)
   print('FIT',held,model_name,'seconds',round(time.monotonic()-t,1),flush=True)
  frame[model_name]=pred
 predictions.append(frame)
P=pd.concat(predictions,ignore_index=True)
assert len(P)==len(D) and not P.duplicated(KEYS).any()
assert P[['M0_mean',*FEATURE_SETS]].notna().all().all()
P.to_parquet(RUN/'01_paired_LOFO_predictions.parquet',index=False)
pd.DataFrame(audit).to_csv(RUN/'01_training_fold_sizes.csv',index=False)
display(pd.DataFrame(audit))
print(f'All held-out predictions cached on {len(D):,} native V5 reference cells.')


In [ ]:
MODELS=['M0_mean',*FEATURE_SETS]
def metrics(g,col):
 obs=g.target_fraction.to_numpy(float)
 pred=g[col].to_numpy(float)
 err=pred-obs
 var=float(np.mean((obs-np.mean(obs))**2))
 rho=spearmanr(obs,pred).statistic if np.std(pred)>1e-12 and np.std(obs)>1e-12 else np.nan
 return dict(n=len(g),R2=float(1-np.mean(err**2)/var) if var>0 else np.nan,
             MAE_pp=float(100*np.mean(np.abs(err))),
             RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
             bias_pp=float(100*np.mean(err)),
             spearman_rho=float(rho) if np.isfinite(rho) else np.nan)
PER=pd.DataFrame([dict(site=forest,model=model,**metrics(g,model))
 for forest,g in P.groupby('site') for model in MODELS])
PER.to_csv(RUN/'02_continuous_by_forest.csv',index=False)
SUMMARY=[]
for model in MODELS:
 sub=PER[PER.model.eq(model)]
 # Global equal-forest R² with pooled equal-forest target variance.
 means=P.groupby('site').target_fraction.mean()
 v=np.mean([np.mean((g.target_fraction-means[s])**2)+(means[s]-means.mean())**2
      for s,g in P.groupby('site')])
 mse=np.mean([np.mean((g[model]-g.target_fraction)**2) for s,g in P.groupby('site')])
 SUMMARY.append(dict(model=model,n_cells=len(P),n_forests=len(SITES),
  equal_forest_R2=float(1-mse/v) if v>0 else np.nan,
  equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
  equal_forest_RMSE_pp=float(np.sqrt(np.mean(sub.RMSE_pp**2))),
  equal_forest_bias_pp=float(sub.bias_pp.mean()),
  mean_forest_rho=float(sub.spearman_rho.mean())))
SUMMARY=pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN/'02_equal_forest_summary.csv',index=False)
DISC=[]
for (forest,model),g in (( (f,m),q) for f,q in P.groupby('site') for m in MODELS):
 obs=g.target_fraction.to_numpy(float)
 pred=g[model].to_numpy(float)
 for t in THRESHOLDS:
  true=(obs>=t).astype(int)
  npos=int(true.sum());nneg=len(true)-npos
  auc=roc_auc_score(true,pred) if npos and nneg and np.unique(pred).size>1 else np.nan
  ap=average_precision_score(true,pred) if npos and nneg else np.nan
  DISC.append(dict(site=forest,model=model,threshold=t,
       n=len(g),positive_n=npos,negative_n=nneg,
       prevalence=npos/len(g),ROC_AUC=auc,average_precision=ap))
DISC=pd.DataFrame(DISC)
DISC.to_csv(RUN/'02_abundance_detection_by_forest.csv',index=False)
DISPLAY_DISC=(DISC.groupby(['model','threshold']).agg(
 n_forests_evaluable=('ROC_AUC','count'),
 mean_AUC=('ROC_AUC','mean'),
 mean_AP=('average_precision','mean'),
 mean_prevalence=('prevalence','mean')).reset_index())
DISPLAY_DISC.to_csv(RUN/'02_equal_forest_detection_summary.csv',index=False)
print('Equal-forest continuous-cover scores:')
display(SUMMARY.round(3))
print('Abundance discrimination (undefined sites omitted, counts explicit):')
display(DISPLAY_DISC.round(3))
print('Important: high-cover detection is undefined for forests without high-cover examples.')


## 04 — Spectral contrasts, detection and prediction figures

In [ ]:
import matplotlib
from matplotlib.lines import Line2D
# A: Within-forest annual S2 contrast. Use percentiles for robustness.
low=.05;high=.40
rows=[]
for forest,g in D.groupby('site'):
 lo=g[g.target_fraction<=low]
 hi=g[g.target_fraction>=high]
 if len(lo)<10 or len(hi)<10:
  print('Spectral contrast not estimable',forest,'low',len(lo),'high',len(hi));continue
 for i,c in enumerate(ANNUAL,1):
  lv=float(lo[c].median());hv=float(hi[c].median())
  rows.append(dict(site=forest,band=c,band_order=i,n_low=len(lo),
                   n_high=len(hi),median_low=lv,median_high=hv,
                   difference=hv-lv))
SPECTRA=pd.DataFrame(rows,columns=['site','band','band_order','n_low','n_high','median_low','median_high','difference'])
SPECTRA.to_csv(RUN/'03_within_forest_annual_S2_spectra.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=SPECTRA[SPECTRA.site.eq(forest)]
 if len(g):
  ax.plot(g.band_order,g.median_low,'o-',label='≤5% UAV Musanga')
  ax.plot(g.band_order,g.median_high,'o-',label='≥40% UAV Musanga')
  ax.set_title(f'{forest} | low n={g.n_low.iloc[0]}, high n={g.n_high.iloc[0]}')
 else:
  ax.set_title(f'{forest} | too few high/low cells')
 ax.set(xlabel='Annual S2 source band number',ylabel='Median annual band value')
 ax.grid(alpha=.15)
axs.flat[0].legend(fontsize=8)
fig.suptitle('A · Within-forest spectral associations (not species-specific identification)')
fig.savefig(RUN/'03A_S2_within_forest_low_high_spectra.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)
# B: For each forest show S2 vs embeddings+S2; cap plots only, not metrics.
fig,axs=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=P[P.site.eq(forest)].copy()
 if len(g)>1300:g=g.sample(n=1300,random_state=SEED)
 ax.scatter(g.target_fraction*100,g.M1_S2*100,s=9,alpha=.3,label='S2')
 ax.scatter(g.target_fraction*100,g.M3_S2_EMB*100,s=9,alpha=.3,label='S2 + embeddings')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlabel='UAV Musanga cover (%)',
        ylabel='Held-out satellite prediction (%)',
        xlim=(-2,102),ylim=(-2,102))
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('B · Strict leave-one-forest-out prediction (matched reference cells)')
fig.savefig(RUN/'03B_heldout_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# C: Mean AUC by cover threshold and number of forests actually evaluable.
fig,ax=plt.subplots(figsize=(8,5))
for model in MODELS[1:]:
 q=DISPLAY_DISC[DISPLAY_DISC.model.eq(model)].sort_values('threshold')
 ax.plot(q.threshold*100,q.mean_AUC,marker='o',label=model)
 for v in q.itertuples():
  if np.isfinite(v.mean_AUC):
   ax.annotate(f'n={v.n_forests_evaluable}',(v.threshold*100,v.mean_AUC),
               xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
ax.axhline(.5,ls='--',color='gray',lw=1,label='Chance AUC')
ax.set(xlabel='UAV Musanga cover threshold (%)',ylabel='Forest-mean held-out ROC-AUC',
       ylim=(0,1.05),xticks=[5,20,40],
       title='C · Ability to distinguish Musanga-abundant patches')
ax.legend(fontsize=9);ax.grid(alpha=.15)
fig.savefig(RUN/'03C_abundance_discrimination.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Three figures saved:',RUN)


In [ ]:
# 10 original annual reflectance bands (verified upstream Earth Engine selection order)
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
S2_WAVELENGTH_NM=np.array([492,560,665,833,704,740,783,865,1610,2190])
S2_ORDER=np.argsort(S2_WAVELENGTH_NM)
assert len(ANNUAL)==15 and len(QUARTER)==44
REFLECTANCE=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
INDICES=[f'RAW_ANNUAL__{i:02d}' for i in range(11,15)]
CLEAR_ANNUAL='RAW_ANNUAL__15'
Q_REFLECTANCE={q:[f'S2_QUARTERLY_3YR__{(q-1)*11+i:02d}' for i in range(1,11)] for q in range(1,5)}
assert set(REFLECTANCE+INDICES+[CLEAR_ANNUAL])==set(ANNUAL)
assert set(sum(Q_REFLECTANCE.values(),[])+Q_COUNTS)==set(QUARTER)
# Plot median and interquartile envelope of raw reflectance; no modelling in this block.
spectral_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(15,10),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast\\nlow={len(low)}, high={len(high)}',
          ha='center',va='center',transform=ax.transAxes)
  ax.set_title(f'{forest} — not estimable')
  continue
 for subset,label,color in [(low,'≤5% Musanga','steelblue'),(high,'≥40% Musanga','darkorange')]:
  arr=subset[REFLECTANCE].to_numpy(float)[:,S2_ORDER]
  wl=S2_WAVELENGTH_NM[S2_ORDER]
  med=np.median(arr,axis=0);q25=np.percentile(arr,25,axis=0);q75=np.percentile(arr,75,axis=0)
  ax.plot(wl,med,'o-',label=f'{label}, n={len(subset)}',color=color)
  ax.fill_between(wl,q25,q75,alpha=.16,color=color)
  for j,k in enumerate(S2_ORDER):
   spectral_rows.append(dict(site=forest,group=label,band=S2_BANDS[k],
     wavelength_nm=int(S2_WAVELENGTH_NM[k]),n=len(subset),
     median=float(med[j]),q25=float(q25[j]),q75=float(q75[j])))
 ax.set(xlabel='Approximate central wavelength (nm)',
        ylabel='S2 annual median surface reflectance',title=forest)
 ax.legend(fontsize=8);ax.grid(alpha=.12)
fig.suptitle('True annual S2 reflectance: low vs high 50 m Musanga cover (median ± IQR)')
fig.savefig(RUN/'05A_annual_S2_true_reflectance_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(spectral_rows).to_csv(RUN/'05A_annual_S2_true_reflectance_IQR.csv',index=False)
# Seasonal reflectance: 3-year pooled quarter medians, NOT single-year snapshots.
seasonal_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast: high={len(high)}',ha='center',va='center',transform=ax.transAxes)
  ax.set_title(forest);continue
 for q in range(1,5):
  for subset,name in [(low,'low'),(high,'high')]:
   # NDVI is computed from quarter median reflectances (B8 = index 4; B4 = index 3).
   nir=subset[Q_REFLECTANCE[q][3]].to_numpy(float)
   red=subset[Q_REFLECTANCE[q][2]].to_numpy(float)
   ndvi=(nir-red)/np.maximum(nir+red,1e-8)
   med=float(np.median(ndvi))
   seasonal_rows.append(dict(site=forest,quarter=q,class_name=name,
                              n=len(subset),median_NDVI=med,
                              q25_NDVI=float(np.percentile(ndvi,25)),
                              q75_NDVI=float(np.percentile(ndvi,75))))
 for name,color in [('low','steelblue'),('high','darkorange')]:
  a=pd.DataFrame(seasonal_rows)
  a=a[(a.site==forest)&(a.class_name==name)].sort_values('quarter')
  ax.plot(a.quarter,a.median_NDVI,'o-',color=color,label=name)
  ax.fill_between(a.quarter.to_numpy(float),a.q25_NDVI.to_numpy(float),
                  a.q75_NDVI.to_numpy(float),alpha=.15,color=color)
 ax.set(title=forest,xlabel='Pooled 3-year calendar quarter',ylabel='Quarterly median-reflectance NDVI',
        xticks=[1,2,3,4])
 ax.grid(alpha=.15);ax.legend()
fig.suptitle('Seasonal spectral association: quarter medians pooled across three years')
fig.savefig(RUN/'05B_pooled_quarterly_NDVI_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(seasonal_rows).to_csv(RUN/'05B_pooled_quarterly_NDVI_IQR.csv',index=False)
print('Spectral plots complete. Overlap of distributions is expected in mixed 50 m canopies.')


## 05 — Temporal and predictor-family LOFO ablation

In [ ]:
ABLATION={
 'A_reflectance_only':REFLECTANCE,
 'B_annual_plus_indices':REFLECTANCE+INDICES,
 'C_pooled_quarters_only':sum(Q_REFLECTANCE.values(),[]),
 'D_annual_plus_quarters':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[]),
 'E_embeddings_only':EMB,
 'F_all_plus_embeddings':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[])+EMB
}
ABL_CFG={'version':'NB08_S2_temporal_ablation_v1','seed':SEED,'model':MODEL,
        'train_cap':CAP_PER_TRAIN_FOREST,'features':ABLATION,
        'source':SOURCE_SHA}
ABL_ID=hashlib.sha256(json.dumps(ABL_CFG,sort_keys=True).encode()).hexdigest()[:12]
ABL_DIR=OUT/'18_S2_temporal_ablation'/ABL_ID
ABL_DIR.mkdir(parents=True,exist_ok=True)
(ABL_DIR/'config.json').write_text(json.dumps(ABL_CFG,indent=2))
rows=[];pred_frames=[]
for forest in SITES:
 test=np.flatnonzero(site==forest)
 train=balanced_draw(np.flatnonzero(site!=forest),forest)
 part=D.iloc[test][KEYS+['target_fraction']].copy().reset_index(drop=True)
 for variant,features in ABLATION.items():
  cache=ABL_DIR/f'{forest}_{variant}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test'],test)
    predict=z['pred'].astype(float)
   print('CACHE',forest,variant,flush=True)
  else:
   start=time.monotonic()
   fit=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   fit.fit(D.iloc[train][features].to_numpy(np.float32),y[train])
   predict=np.clip(fit.predict(D.iloc[test][features].to_numpy(np.float32)),0,1)
   with open(cache.with_suffix('.partial'),'wb') as f:
    np.savez_compressed(f,test=test,pred=predict.astype(np.float32))
   os.replace(cache.with_suffix('.partial'),cache)
   print('FIT',forest,variant,round(time.monotonic()-start,1),'seconds',flush=True)
  part[variant]=predict
  err=predict-y[test]
  rows.append(dict(site=forest,variant=variant,n=len(test),n_features=len(features),
     MAE_pp=float(100*np.mean(abs(err))),MSE=float(np.mean(err**2)),
     bias_pp=float(100*np.mean(err)),
     R2=float(1-np.mean(err**2)/np.var(y[test])) if np.var(y[test])>0 else np.nan))
 pred_frames.append(part)
ABL_P=pd.concat(pred_frames,ignore_index=True)
ABL_P.to_parquet(ABL_DIR/'06_all_paired_LOFO_predictions.parquet',index=False)
ABL_PER=pd.DataFrame(rows)
ABL_PER.to_csv(ABL_DIR/'06_per_forest.csv',index=False)
forest_mean=D.groupby('site').target_fraction.mean().mean()
target_variance=float(np.mean([
 np.mean((q.target_fraction.to_numpy(float)-forest_mean)**2)
 for _,q in D.groupby('site')]))
summary=ABL_PER.groupby('variant').agg(n_forests=('site','nunique'),
  mean_MAE_pp=('MAE_pp','mean'),mean_MSE=('MSE','mean'),
  mean_bias_pp=('bias_pp','mean'),mean_per_forest_R2=('R2','mean'))
summary['equal_forest_R2']=1-summary.mean_MSE/target_variance
summary['equal_forest_RMSE_pp']=100*np.sqrt(summary.mean_MSE)
summary=summary.reset_index()
summary.to_csv(ABL_DIR/'06_equal_forest_ablation_summary.csv',index=False)
display(summary[['variant','n_forests','mean_MAE_pp','equal_forest_R2',
                 'equal_forest_RMSE_pp','mean_bias_pp']].round(3))
display(ABL_PER.pivot(index='site',columns='variant',values='MAE_pp').round(2))
fig,ax=plt.subplots(figsize=(11,5),layout='constrained')
ax.bar(summary.variant,summary.equal_forest_R2)
ax.axhline(0,color='gray',lw=.8)
ax.tick_params(axis='x',rotation=28)
ax.set(ylabel='Equal-forest R²',title='Does seasonal S2 add transfer beyond annual reflectance and indices?')
fig.savefig(ABL_DIR/'06_S2_annual_vs_seasonal_LOFO.png',dpi=180)
plt.show();plt.close(fig)
print('Annual-versus-seasonal LOFO results:',ABL_DIR)


## 06 — Native scale-specific annual Sentinel-2 LOFO (10, 25, 30, 50, 75, 100 m)

In [ ]:
SCALE_SEED=20261120
MAX_TRAIN_PER_FOREST=1500
MAX_TEST_PER_FOREST=15000 # deterministic cap for rapid, unbiased within-forest sample
SCALE_FEATURES=[f'S2_{j:02d}' for j in range(1,15)]
SCALE_MODELS=['annual_reflectance_indices']
SCALE_MODEL_ARGS=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                      min_samples_leaf=35,l2_regularization=2,early_stopping=False)
scale_metrics=[];all_out=[]
for scale in SCALES:
 samples={s:pd.read_parquet(SCALE_RUN/f'{s}_{scale}m.parquet') for s in SITES}
 for held in SITES:
  rng=np.random.default_rng(SCALE_SEED+scale*31+sum(map(ord,held)))
  target=samples[held]
  test_ids=np.arange(len(target))
  if len(test_ids)>MAX_TEST_PER_FOREST:
   test_ids=np.sort(rng.choice(test_ids,MAX_TEST_PER_FOREST,replace=False))
  training=[]
  for s in SITES:
   if s==held:continue
   other=samples[s]
   # Forest balanced training; select up to cap, stratified by abundance using training labels only.
   ids=np.arange(len(other))
   strata=np.digitize(other.target_fraction.to_numpy(),[.10,.20,.40])
   perbin=[ids[strata==i] for i in range(4)]
   pick=[]
   for b in perbin:
    if len(b):
     n=min(len(b),max(1,MAX_TRAIN_PER_FOREST//4))
     pick.extend(rng.choice(b,n,replace=False).tolist())
   remaining=MAX_TRAIN_PER_FOREST-len(pick)
   if remaining>0:
    rest=np.setdiff1d(ids,np.asarray(pick,int))
    if len(rest):pick.extend(rng.choice(rest,min(remaining,len(rest)),replace=False).tolist())
   training.append(other.iloc[np.asarray(sorted(pick),int)])
  fit_data=pd.concat(training,ignore_index=True)
  output=SCALE_RUN/f'09_LOFO_{scale}m_{held}.parquet'
  if output.is_file():
   pred_frame=pd.read_parquet(output)
   assert len(pred_frame)==len(test_ids)
   print('CACHE LOFO',scale,held,flush=True)
  else:
   model=HistGradientBoostingRegressor(random_state=SCALE_SEED,**SCALE_MODEL_ARGS)
   model.fit(fit_data[SCALE_FEATURES].to_numpy(np.float32),
             fit_data.target_fraction.to_numpy(np.float32))
   te=target.iloc[test_ids]
   pred=np.clip(model.predict(te[SCALE_FEATURES].to_numpy(np.float32)),0,1)
   pred_frame=te[['site','scale_m','row','col','x','y','target_fraction']].copy().reset_index(drop=True)
   pred_frame['prediction']=pred
   pred_frame.to_parquet(output,index=False)
   print('FIT LOFO',scale,held,'test',len(pred_frame),flush=True)
  ytrue=pred_frame.target_fraction.to_numpy(float)
  ypred=pred_frame.prediction.to_numpy(float)
  err=ypred-ytrue
  var=np.var(ytrue)
  scale_metrics.append(dict(scale_m=scale,site=held,n_test=len(pred_frame),
    n_train=len(fit_data),target_mean_pp=float(np.mean(ytrue)*100),
    target_sd_pp=float(np.std(ytrue)*100),
    MAE_pp=float(np.mean(abs(err))*100),RMSE_pp=float(np.sqrt(np.mean(err**2))*100),
    bias_pp=float(np.mean(err)*100),
    R2=float(1-np.mean(err**2)/var) if var>1e-12 else np.nan))
  all_out.append(pred_frame)
SCALE_PER=pd.DataFrame(scale_metrics)
SCALE_PER.to_csv(SCALE_RUN/'09_LOFO_per_forest_scale.csv',index=False)
# Compare equal-forest scores at each spatial support, incl. correct pooled variance.
equal=[]
for scale,g in SCALE_PER.groupby('scale_m'):
 pred=pd.concat([p for p in all_out if int(p.scale_m.iloc[0])==scale],ignore_index=True)
 means=pred.groupby('site').target_fraction.mean()
 overall=float(means.mean())
 variance=np.mean([np.mean((q.target_fraction.to_numpy()-overall)**2)
                       for _,q in pred.groupby('site')])
 mse=np.mean([np.mean((q.prediction.to_numpy()-q.target_fraction.to_numpy())**2)
                   for _,q in pred.groupby('site')])
 equal.append(dict(scale_m=scale,forests=len(g),
    equal_forest_R2=float(1-mse/variance) if variance>0 else np.nan,
    equal_forest_MAE_pp=float(g.MAE_pp.mean()),
    equal_forest_RMSE_pp=float(np.sqrt(np.mean(g.RMSE_pp**2))),
    equal_forest_target_SD_pp=float(np.sqrt(variance)*100),
    min_site_R2=float(g.R2.min()),MOS_R2=float(g[g.site.eq('MOS')].R2.iloc[0])))
SCALE_SUM=pd.DataFrame(equal).sort_values('scale_m')
SCALE_SUM.to_csv(SCALE_RUN/'09_LOFO_equal_forest_scale_summary.csv',index=False)
display(SCALE_SUM.round(3))
display(SCALE_PER.pivot(index='scale_m',columns='site',values='R2').round(3))
print('Scale prediction outputs:',SCALE_RUN)


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.8),layout='constrained')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_R2,'o-',label='Equal forest')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.MOS_R2,'s--',label='MOS only')
ax[0].set(ylabel='LOFO R²',title='Geographic transfer')
ax[0].axhline(0,ls=':',c='grey');ax[0].legend()
ax[1].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_MAE_pp,'o-')
ax[1].set(ylabel='MAE (percentage points)',title='Absolute abundance error')
ax[2].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_target_SD_pp,'o-')
ax[2].set(ylabel='Target SD (percentage points)',title='Target variability changes with scale')
for a in ax:
 a.set_xlabel('Aggregation scale (m)')
 a.set_xticks(SCALE_SUM.scale_m.to_numpy())
 a.grid(alpha=.15)
fig.suptitle('Native V3.1 UAV teacher + annual S2: honest multiresolution LOFO')
fig.savefig(SCALE_RUN/'10_multiresolution_LOFO_three_metrics.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
fig,axes=plt.subplots(2,2,figsize=(11,9),layout='constrained')
for axis,forest in zip(axes.flat,SITES):
 q=SCALE_PER[SCALE_PER.site.eq(forest)].sort_values('scale_m')
 axis.plot(q.scale_m,q.R2,'o-',label='R²')
 axis.axhline(0,ls='--',lw=.8,color='gray')
 axis.set(title=forest,xlabel='Aggregation scale (m)',ylabel='Held-out forest R²')
 axis.set_xticks(q.scale_m)
 axis.grid(alpha=.15)
fig.savefig(SCALE_RUN/'10_multiresolution_per_forest_R2.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Interpretation: choose a stable resolution based on MAE, forest transfer, and ecological map utility;')
print('avoid choosing scale by one inflated pooled R².')


## 07 — 30/50m transfer diagnostics and spatial block uncertainty

In [ ]:
SCALE_DIAG=SCALE_RUN/'transfer_diagnostics';SCALE_DIAG.mkdir(parents=True,exist_ok=True)
V19=[]
for scale in [30,50]:
 for forest in SITES:
  path=SCALE_RUN/f'09_LOFO_{scale}m_{forest}.parquet'
  assert path.exists(),('Missing NB08 held-out predictions',path)
  d=pd.read_parquet(path)
  assert d.site.eq(forest).all() and d.target_fraction.between(0,1).all()
  assert d.prediction.between(0,1).all()
  d=d.copy();d['scale_m']=scale;V19.append(d)
V19=pd.concat(V19,ignore_index=True)
V19['cover_bin']=pd.cut(V19.target_fraction,[-.001,.05,.20,.40,.60,1.001],labels=['0-5%','5-20%','20-40%','40-60%','60-100%'],include_lowest=True)
metrics19=[];bins19=[];equal19=[]
for (scale,forest),g in V19.groupby(['scale_m','site']):
 y=g.target_fraction.to_numpy(float);p=g.prediction.to_numpy(float);e=p-y;high=y>=.4
 metrics19.append(dict(scale_m=scale,site=forest,n=len(g),R2=1-np.mean(e**2)/np.var(y) if np.var(y)>0 else np.nan,
  MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),bias_pp=100*np.mean(e),
  target_SD_pp=100*np.std(y),n_high=int(high.sum()),high_bias_pp=100*np.mean(e[high]) if high.any() else np.nan,
  high_recall=float(np.mean(p[high]>=.4)) if high.any() else np.nan))
 for b,z in g.groupby('cover_bin',observed=True):
  bins19.append(dict(scale_m=scale,site=forest,cover_bin=str(b),n=len(z),
    observed_pp=100*z.target_fraction.mean(),predicted_pp=100*z.prediction.mean(),
    bias_pp=100*(z.prediction-z.target_fraction).mean()))
for scale,g in V19.groupby('scale_m'):
 forests=[v for _,v in g.groupby('site')]
 mean_ref=np.mean([f.target_fraction.mean() for f in forests])
 mse=np.mean([np.mean((f.prediction-f.target_fraction)**2) for f in forests])
 variance=np.mean([np.mean((f.target_fraction-mean_ref)**2) for f in forests])
 equal19.append(dict(scale_m=scale,n_forests=len(forests),forest_equal_R2=1-mse/variance,
  forest_equal_MAE_pp=100*np.mean([np.mean(abs(f.prediction-f.target_fraction)) for f in forests]),
  forest_equal_RMSE_pp=100*np.sqrt(mse),forest_equal_target_SD_pp=100*np.sqrt(variance)))
MET19=pd.DataFrame(metrics19);BIN19=pd.DataFrame(bins19);EQ19=pd.DataFrame(equal19)
MET19.to_csv(SCALE_DIAG/'19_LOFO_per_forest_30_50m.csv',index=False)
BIN19.to_csv(SCALE_DIAG/'19_LOFO_cover_bins_30_50m.csv',index=False)
EQ19.to_csv(SCALE_DIAG/'19_LOFO_forest_equal_30_50m.csv',index=False)
display(EQ19.round(3));display(MET19.round(3));display(BIN19.round(2))
fig,axes=plt.subplots(1,2,figsize=(13,5),sharex=True,sharey=True,layout='constrained')
for ax,scale in zip(axes,[30,50]):
 for i,forest in enumerate(SITES):
  g=V19[(V19.scale_m==scale)&(V19.site==forest)]
  sample=g.sample(n=min(len(g),2000),random_state=SEED17+scale+i)
  ax.scatter(sample.target_fraction*100,sample.prediction*100,s=6,alpha=.15,label=f'{forest} ({len(g):,})')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(xlim=(0,100),ylim=(0,100),xlabel='UAV reference cover (%)',ylabel='Satellite-predicted cover (%)',title=f'{scale} m — LOFO')
 ax.legend(fontsize=8)
fig.savefig(SCALE_DIAG/'19_balanced_display_30_50m.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)
fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 for scale in [30,50]:
  a=BIN19[(BIN19.scale_m==scale)&(BIN19.site==forest)]
  ax.plot(a.observed_pp,a.predicted_pp,'o-',label=f'{scale} m')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlim=(0,100),ylim=(0,100),xlabel='Observed cover bin mean (%)',ylabel='Predicted cover bin mean (%)')
 ax.legend()
fig.savefig(SCALE_DIAG/'19_cover_bin_calibration.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)


In [ ]:
rows20=[]
for scale in [30,50]:
 byforest=[];nall=0
 for forest in SITES:
  g=V19[(V19.scale_m==scale)&(V19.site==forest)].copy()
  if not {'x','y'}.issubset(g.columns):
   coords=pd.read_parquet(SCALE_RUN/f'{forest}_{scale}m.parquet',columns=['site','row','col','x','y'])
   g=g.merge(coords,on=['site','row','col'],validate='one_to_one')
  assert g[['x','y']].notna().all().all()
  g['block500']=g.x.floordiv(500).astype('int64').astype(str)+'_'+g.y.floordiv(500).astype('int64').astype(str)
  agg=g.assign(abs_error=(g.prediction-g.target_fraction).abs(),error=g.prediction-g.target_fraction).groupby('block500').agg(n=('error','size'),ae=('abs_error','sum'),err=('error','sum'))
  n=len(agg);nall+=n
  rng=np.random.default_rng(SEED17+scale+sum(map(ord,forest)))
  vals=agg[['n','ae','err']].to_numpy(float)
  draw=rng.integers(0,n,(1000,n))
  total=vals[draw].sum(axis=1)
  mae=100*total[:,1]/total[:,0];bias=100*total[:,2]/total[:,0]
  byforest.append(mae)
  rows20.append(dict(scale_m=scale,site=forest,n_blocks=n,MAE_ci_low_pp=np.quantile(mae,.025),MAE_ci_high_pp=np.quantile(mae,.975),
     bias_ci_low_pp=np.quantile(bias,.025),bias_ci_high_pp=np.quantile(bias,.975)))
 equal_dist=np.mean(byforest,axis=0)
 rows20.append(dict(scale_m=scale,site='forest_equal',n_blocks=nall,MAE_ci_low_pp=np.quantile(equal_dist,.025),
    MAE_ci_high_pp=np.quantile(equal_dist,.975),bias_ci_low_pp=np.nan,bias_ci_high_pp=np.nan))
CI20=pd.DataFrame(rows20);CI20.to_csv(SCALE_DIAG/'20_spatial_bootstrap_uncertainty.csv',index=False)
display(CI20.round(3))
print('Interpretation: a significant B8 contrast requires balance-qualified matches; do not claim species-specificity.')
print('Resolution choice: consider forest-equal skill, MOS high-cover bias, target support, and spatial block intervals together.')
print('New output folder:',SCALE_DIAG)


## 08 — Five-fold spatial-block validation within represented forests; matched LOFO

In [ ]:
# SECTION 21 — spatially blocked 50 m CV, forest-balanced training, common-cell LOFO comparison
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression
import hashlib,json,os
assert 'D4' in globals() and 'P' in globals(), 'Run 00V5 and Sections 01–02 first'
assert D.equals(D4), 'Active target is not the Section 00V5 V5 binary-cover reference'
S21=['Baego','MOS','Yangambi','Yoko']
K21=['site','mosaic','row','col']
F21={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':list(S2)+list(EMB)}
B21=500.0
BUFFER21=100.0
FOLDS21=5
CAP21=1500
SEED21=20261123
M21=D.copy().reset_index(drop=True)
assert not M21.duplicated(K21).any()
assert M21.target_fraction.between(0,1).all()
assert set(S21)==set(M21.site.unique())
assert M21[['x','y']+list(set(sum(F21.values(),[])))].notna().all().all()
M21['block_x21']=np.floor(M21.x/B21).astype('int64')
M21['block_y21']=np.floor(M21.y/B21).astype('int64')
M21['block21']=(M21.site.astype(str)+'_'+M21.block_x21.astype(str)+'_'+M21.block_y21.astype(str))
M21['fold21']=-1
# Block-count balancing by cell numbers, without examining response or predictors.
rng21=np.random.default_rng(SEED21)
for forest in S21:
    ix=np.flatnonzero(M21.site.to_numpy()==forest)
    counts=M21.iloc[ix].groupby('block21').size()
    assert len(counts)>=FOLDS21,(forest,'Not enough spatial blocks',len(counts))
    shuffled=rng21.permutation(len(counts))
    blocks=counts.index.to_numpy()[shuffled]
    sizes=counts.to_numpy()[shuffled]
    order=np.argsort(-sizes,kind='stable')
    fold_size=np.zeros(FOLDS21,dtype=int)
    mapping={}
    for k in order:
        fold=int(np.argmin(fold_size))
        mapping[blocks[k]]=fold
        fold_size[fold]+=int(sizes[k])
    M21.loc[ix,'fold21']=M21.iloc[ix].block21.map(mapping).to_numpy(int)
assert M21.fold21.ge(0).all()
assert M21.groupby('block21').fold21.nunique().eq(1).all()
source_key21=hashlib.sha256(pd.util.hash_pandas_object(
    M21[K21+['x','y','target_fraction','fold21']],index=False).values.tobytes()).hexdigest()
CFG21=dict(version='v5_spatial_block_balanced_50m_v1',
    source_sha=SOURCE_SHA,fold_sha=source_key21,block_m=B21,buffer_m=BUFFER21,
    folds=FOLDS21,forest_cap=CAP21,seed=SEED21,models=MODEL,
    feature_sets={k:len(v) for k,v in F21.items()},
    target='NB07_V5_binary_fraction')
RUN21=OUT/'25_NB08_V5_spatial_block_50m'/hashlib.sha256(
    json.dumps(CFG21,sort_keys=True).encode()).hexdigest()[:14]
RUN21.mkdir(parents=True,exist_ok=True)
cfg_file21=RUN21/'21_config.json'
if cfg_file21.exists():
    assert json.loads(cfg_file21.read_text())==CFG21,'Run configuration mismatch'
else:
    cfg_file21.write_text(json.dumps(CFG21,indent=2))
M21[K21+['x','y','block21','fold21','target_fraction']].to_parquet(
    RUN21/'21_spatial_folds.parquet',index=False)
print('Spatial blocks by forest:',M21.groupby('site').block21.nunique().to_dict())
print('Cells per fold and forest:');display(M21.groupby(['site','fold21']).size().unstack(fill_value=0))
# Each test fold has all four forests represented. Buffer removes any training
# cell closer than 100 m to a held-out cell in the SAME forest.
P21=M21[K21+['x','y','target_fraction','fold21','block21']].copy()
for name in F21:P21[name]=np.nan
AUDIT21=[]
for fold in range(FOLDS21):
    test_ix=np.flatnonzero(M21.fold21.to_numpy()==fold)
    training=[]
    audit={'fold':fold,'n_test':len(test_ix)}
    for forest in S21:
        pool=np.flatnonzero((M21.site.to_numpy()==forest)&(M21.fold21.to_numpy()!=fold))
        held=test_ix[M21.site.to_numpy()[test_ix]==forest]
        test_points=M21.iloc[held][['x','y']].to_numpy(float)
        train_points=M21.iloc[pool][['x','y']].to_numpy(float)
        if len(test_points):
            far=cKDTree(test_points).query(train_points,k=1)[0]>=BUFFER21
            pool=pool[far]
        assert len(pool)>0,(fold,forest,'Buffer leaves no training cells')
        rng=np.random.default_rng(SEED21+fold*101+sum(map(ord,forest)))
        if len(pool)>CAP21:
            ytrain=M21.iloc[pool].target_fraction.to_numpy(float)
            strata=np.digitize(ytrain,[.10,.20,.40])
            take=[]
            for k in range(4):
                group=pool[strata==k]
                if len(group):
                    n=min(len(group),CAP21//4)
                    take.extend(rng.choice(group,n,replace=False).tolist())
            if len(take)<CAP21:
                leftover=np.setdiff1d(pool,np.array(take,dtype=int),assume_unique=False)
                take.extend(rng.choice(leftover,min(CAP21-len(take),len(leftover)),replace=False).tolist())
            pool=np.array(sorted(take),dtype=int)
        training.extend(pool.tolist())
        audit[forest+'_train']=len(pool)
        audit[forest+'_test']=len(held)
    train_ix=np.asarray(training,dtype=int)
    assert not np.intersect1d(train_ix,test_ix).size
    assert set(M21.iloc[train_ix].block21).isdisjoint(set(M21.iloc[test_ix].block21))
    for name,features in F21.items():
        cache21=RUN21/f'fold{fold}_{name}.npz'
        if cache21.is_file():
            with np.load(cache21,allow_pickle=False) as z:
                assert z['source_sha'].item()==source_key21
                assert np.array_equal(z['test_ix'],test_ix)
                pred=z['pred']
            print('CACHE 21',fold,name,flush=True)
        else:
            model=HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
            model.fit(M21.iloc[train_ix][features].to_numpy(np.float32),
                      M21.iloc[train_ix].target_fraction.to_numpy(np.float32))
            pred=np.clip(model.predict(M21.iloc[test_ix][features].to_numpy(np.float32)),0,1)
            temp=cache21.with_suffix('.partial')
            with open(temp,'wb') as fd:
                np.savez_compressed(fd,source_sha=source_key21,test_ix=test_ix,pred=pred)
            os.replace(temp,cache21)
            print('FIT 21',fold,name,'test',len(test_ix),flush=True)
        P21.loc[test_ix,name]=pred
    AUDIT21.append(audit)
assert P21[list(F21)].notna().all().all()
A21=pd.DataFrame(AUDIT21)
A21.to_csv(RUN21/'21_fold_training_audit.csv',index=False)
P21.to_parquet(RUN21/'21_spatial_block_OOF_predictions.parquet',index=False)
display(A21)

def summaries21(frame,pred_names,validation):
    rows=[]
    for name in pred_names:
        groups=list(frame.groupby('site'))
        overall=float(np.mean([g.target_fraction.mean() for _,g in groups]))
        pooled_mse=np.mean([np.mean((g[name]-g.target_fraction)**2) for _,g in groups])
        pooled_var=np.mean([np.mean((g.target_fraction-overall)**2) for _,g in groups])
        site_rows=[]
        for forest,g in groups:
            y=g.target_fraction.to_numpy(float);p=g[name].to_numpy(float)
            e=p-y;hi=y>=.4
            fit=LinearRegression().fit(y.reshape(-1,1),p) if np.var(y)>1e-10 else None
            site_rows.append(dict(validation=validation,model=name,site=forest,n=len(g),
                R2=1-np.mean(e**2)/np.var(y) if np.var(y)>1e-10 else np.nan,
                MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),
                bias_pp=100*np.mean(e),calibration_slope=fit.coef_[0] if fit else np.nan,
                calibration_intercept_pp=100*fit.intercept_ if fit else np.nan,
                high_n=int(hi.sum()),high_bias_pp=100*np.mean(e[hi]) if hi.any() else np.nan,
                high_recall=np.mean(p[hi]>=.4) if hi.any() else np.nan))
        rows.extend(site_rows)
        rows.append(dict(validation=validation,model=name,site='forest_equal',n=len(frame),
            R2=1-pooled_mse/pooled_var if pooled_var>0 else np.nan,
            MAE_pp=np.mean([r['MAE_pp'] for r in site_rows]),
            RMSE_pp=np.sqrt(np.mean([r['RMSE_pp']**2 for r in site_rows])),
            bias_pp=np.mean([r['bias_pp'] for r in site_rows]),
            calibration_slope=np.nan,calibration_intercept_pp=np.nan,
            high_n=sum(r['high_n'] for r in site_rows),
            high_bias_pp=np.nan,high_recall=np.nan))
    return pd.DataFrame(rows)

# Match LOFO to precisely the same V5 50m geographical keys.
LOFO21=P[K21+list(F21)].copy()
PAIR21=P21.merge(LOFO21,on=K21,validate='one_to_one',suffixes=('_block','_lofo'))
assert len(PAIR21)==len(P21),(len(PAIR21),len(P21))
B21P=PAIR21[K21+['target_fraction']+[name+'_block' for name in F21]].copy()
L21P=PAIR21[K21+['target_fraction']+[name+'_lofo' for name in F21]].copy()
B21P=B21P.rename(columns={name+'_block':name for name in F21})
L21P=L21P.rename(columns={name+'_lofo':name for name in F21})
MET21=pd.concat([summaries21(B21P,list(F21),'spatial_block'),
                 summaries21(L21P,list(F21),'LOFO')],ignore_index=True)
MET21.to_csv(RUN21/'21_matched_LOFO_vs_spatial_block_metrics.csv',index=False)
PAIR21.to_parquet(RUN21/'21_matched_LOFO_vs_spatial_block_predictions.parquet',index=False)
display(MET21[MET21.site.eq('forest_equal')][
    ['validation','model','n','R2','MAE_pp','RMSE_pp','bias_pp']].round(3))
display(MET21[MET21.site.ne('forest_equal')][
    ['validation','model','site','R2','MAE_pp','bias_pp','high_bias_pp','high_recall']].round(3))

# Balanced display is NOT balanced performance estimation. All held-out cells
# contribute to the per-forest scores and each forest has equal weight.
rng=np.random.default_rng(SEED21+77)
draws=[]
for forest in S21:
    sub=PAIR21[PAIR21.site.eq(forest)]
    draws.append(sub.sample(n=min(175,len(sub)),random_state=SEED21+sum(map(ord,forest))))
SHOW21=pd.concat(draws,ignore_index=True)
colors21=dict(zip(S21,['#236e85','#c17930','#619e65','#9461a5']))
fig,axs=plt.subplots(2,3,figsize=(15,9),sharex=True,sharey=True,layout='constrained')
for row,method in enumerate(['block','lofo']):
    for col,model_name in enumerate(F21):
        ax=axs[row,col]
        for forest in S21:
            sub=SHOW21[SHOW21.site.eq(forest)]
            ax.scatter(sub.target_fraction*100,sub[model_name+'_'+method]*100,
                       s=13,alpha=.48,c=colors21[forest],label=forest)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),
               title=('Spatial-block CV' if row==0 else 'LOFO')+' — '+model_name,
               xlabel='UAV Musanga cover (%)',ylabel='Predicted cover (%)')
        ax.grid(alpha=.14)
h,l=axs[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',ncol=4)
fig.savefig(RUN21/'21_balanced_scatter_spatial_block_vs_LOFO.png',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)

bins21=[-.001,.05,.20,.40,.60,1.001]
binlabels21=['0–5','5–20','20–40','40–60','60–100']
cal21=[]
for forest,group in PAIR21.groupby('site'):
    for method in ('block','lofo'):
        for model in F21:
            q=group.copy()
            q['bin']=pd.cut(q.target_fraction,bins21,labels=binlabels21,include_lowest=True)
            for band,g in q.groupby('bin',observed=True):
                cal21.append(dict(site=forest,validation=method,model=model,cover_bin=str(band),
                    n=len(g),observed_pp=100*g.target_fraction.mean(),
                    predicted_pp=100*g[model+'_'+method].mean(),
                    bias_pp=100*(g[model+'_'+method]-g.target_fraction).mean()))
CAL21=pd.DataFrame(cal21)
CAL21.to_csv(RUN21/'21_cover_bin_calibration.csv',index=False)
fig,ax=plt.subplots(figsize=(8,6))
for forest in S21:
    for method,style in [('block','-'),('lofo','--')]:
        part=CAL21[(CAL21.site==forest)&(CAL21.validation==method)&(CAL21.model=='M1_S2')]
        ax.plot(part.observed_pp,part.predicted_pp,marker='o',linestyle=style,
                color=colors21[forest],label=forest+' '+method)
ax.plot([0,100],[0,100],'k:',label='1:1')
ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed Musanga cover (%)',
       ylabel='Mean predicted cover (%)',title='50 m S2: forest-wise cover-bin calibration')
ax.legend(fontsize=8,ncol=2);ax.grid(alpha=.2)
fig.savefig(RUN21/'21_50m_S2_cover_bin_calibration.png',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)
print('Section 21 complete. Outputs:',RUN21)


## 09 — Exploratory abundance weighting, two-stage and leakage-controlled nested calibration

In [ ]:
if RUN_ABUNDANCE_SENSITIVITY:
    # SECTION 23 — abundance-weighting, probabilistic two-stage, nested spatial calibration
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.isotonic import IsotonicRegression
    from scipy.spatial import cKDTree
    assert 'M21' in globals() and 'P21' in globals() and 'RUN21' in globals(), 'Run Section 21 first'
    assert 'summaries21' in globals()
    S23=['Baego','MOS','Yangambi','Yoko']
    SETS23={'S2':list(S2),'S2_EMB':list(S2)+list(EMB)}
    SEED23=20261124
    CAP23=1500
    WEIGHTS23={'moderate':[1.,1.,1.5,2.,3.],
               'strong':[1.,1.,2.,3.,5.]}
    STRATEGIES23=['baseline','moderate','strong','two_stage','nested_isotonic']
    K23=['site','mosaic','row','col']
    assert M21[K23].equals(P21[K23])
    assert len(M21)==len(D) and set(M21.site)==set(S23)
    assert M21.fold21.nunique()==5
    CFG23=dict(version='v5_abundance_spatial_sensitivity_v2',source_sha=SOURCE_SHA,
               folds_hash=source_key21,seed=SEED23,cap=CAP23,
               weights=WEIGHTS23,features={k:len(v) for k,v in SETS23.items()},
               calibration='four inner block folds nested in each outer fold')
    RUN23=OUT/'27_NB08_V5_abundance_sensitivity_50m'/hashlib.sha256(
        json.dumps(CFG23,sort_keys=True).encode()).hexdigest()[:14]
    RUN23.mkdir(parents=True,exist_ok=True)
    p23=RUN23/'23_config.json'
    if p23.exists():assert json.loads(p23.read_text())==CFG23
    else:p23.write_text(json.dumps(CFG23,indent=2))
    Y23=M21.target_fraction.to_numpy(float)
    SITE23=M21.site.to_numpy()
    FOLD23=M21.fold21.to_numpy()
    XY23=M21[['x','y']].to_numpy(float)
    def pick23(pool,forest,fold):
        if len(pool)<=CAP23:return pool
        # Exactly reproduce Section 21's training cap sampling.
        rng=np.random.default_rng(SEED21+fold*101+sum(map(ord,forest)))
        strata=np.digitize(Y23[pool],[.10,.20,.40])
        chosen=[]
        for k in range(4):
            ids=pool[strata==k]
            if len(ids):chosen.extend(rng.choice(ids,min(len(ids),CAP23//4),replace=False).tolist())
        if len(chosen)<CAP23:
            rest=np.setdiff1d(pool,np.array(chosen,dtype=int),assume_unique=False)
            chosen.extend(rng.choice(rest,CAP23-len(chosen),replace=False).tolist())
        return np.array(sorted(chosen),dtype=int)
    def training23(excluded_folds,validation_idx,fold):
        train=[]
        counts={}
        for forest in S23:
            pool=np.flatnonzero((SITE23==forest)&(~np.isin(FOLD23,excluded_folds)))
            val=np.flatnonzero((SITE23==forest)&np.isin(FOLD23,excluded_folds))
            if len(val):
                distance=cKDTree(XY23[val]).query(XY23[pool],k=1)[0]
                pool=pool[distance>=100.]
            assert len(pool)>0,(forest,fold,'No spatially buffered training cells')
            pool=pick23(pool,forest,fold)
            train.extend(pool.tolist())
            counts[forest]=len(pool)
        train=np.asarray(train,dtype=int)
        assert not np.intersect1d(train,validation_idx).size
        assert set(M21.iloc[train].block21).isdisjoint(set(M21.iloc[validation_idx].block21))
        return train,counts
    def balanced_weights23(labels,forests,profile=None):
        weights=np.ones(len(labels),dtype=float)
        if profile is not None:
            bins=np.digitize(labels,[.05,.20,.40,.60])
            weights=np.asarray(WEIGHTS23[profile],dtype=float)[bins]
        for forest in S23:
            mask=forests==forest
            if mask.any():weights[mask]/=weights[mask].sum()
        return weights*len(weights)/len(S23)
    def cached23(path,test_idx,fit_function):
        if path.is_file():
            with np.load(path,allow_pickle=False) as z:
                assert np.array_equal(z['test_idx'],test_idx)
                assert z['source_sha'].item()==source_key21
                return z['prediction']
        prediction=np.asarray(fit_function(),dtype=float)
        assert len(prediction)==len(test_idx) and np.isfinite(prediction).all()
        partial=path.with_suffix('.partial')
        with open(partial,'wb') as fh:
            np.savez_compressed(fh,test_idx=test_idx,source_sha=source_key21,
                                prediction=prediction)
        os.replace(partial,path)
        return prediction
    P23=M21[K23+['x','y','target_fraction','fold21','block21']].copy()
    for fs in SETS23:
        for strategy in STRATEGIES23:P23[fs+'__'+strategy]=np.nan
    audit23=[]
    for outer in range(5):
        test=np.flatnonzero(FOLD23==outer)
        train,counts=training23([outer],test,outer)
        audit23.append(dict(fold=outer,n_test=len(test),**counts))
        for fs,features in SETS23.items():
            baseline_col={'S2':'M1_S2','S2_EMB':'M3_S2_EMB'}[fs]
            base=P21.iloc[test][baseline_col].to_numpy(float)
            P23.loc[test,fs+'__baseline']=base
            Xtr=M21.iloc[train][features].to_numpy(np.float32)
            Xte=M21.iloc[test][features].to_numpy(np.float32)
            yt=Y23[train];st=SITE23[train]
            for scenario in WEIGHTS23:
                def train_weighted23():
                    model=HistGradientBoostingRegressor(random_state=SEED23,**MODEL)
                    model.fit(Xtr,yt,sample_weight=balanced_weights23(yt,st,scenario))
                    return np.clip(model.predict(Xte),0,1)
                f=RUN23/f'{fs}_{scenario}_outer{outer}.npz'
                pred=cached23(f,test,train_weighted23)
                P23.loc[test,fs+'__'+scenario]=pred
                print('DONE',outer,fs,scenario,flush=True)
            def train_two_stage23():
                classes=np.digitize(yt,[.05,.20,.40])
                if len(np.unique(classes))<2:return np.full(len(Xte),float(yt.mean()))
                classifier=HistGradientBoostingClassifier(random_state=SEED23,**MODEL)
                classifier.fit(Xtr,classes,sample_weight=balanced_weights23(yt,st,'moderate'))
                probs=np.zeros((len(Xte),4),float)
                probs[:,classifier.classes_.astype(int)]=classifier.predict_proba(Xte)
                conditional=np.empty_like(probs)
                bounds=[(0,.05),(.05,.20),(.20,.40),(.40,1.)]
                for c,(low,high) in enumerate(bounds):
                    ix=classes==c
                    if ix.sum()>=35 and np.unique(yt[ix]).size>=10:
                        r=HistGradientBoostingRegressor(random_state=SEED23+c,**MODEL)
                        r.fit(Xtr[ix],yt[ix],sample_weight=balanced_weights23(
                            yt[ix],st[ix],'moderate'))
                        conditional[:,c]=np.clip(r.predict(Xte),low,high)
                    else:
                        conditional[:,c]=float(yt[ix].mean()) if ix.any() else (low+high)/2
                return np.clip(np.sum(probs*conditional,axis=1),0,1)
            P23.loc[test,fs+'__two_stage']=cached23(
                RUN23/f'{fs}_two_stage_outer{outer}.npz',test,train_two_stage23)
            print('DONE',outer,fs,'two_stage',flush=True)
            def train_nested_cal23():
                # Outer test observations NEVER participate in inner training or calibration.
                inner_predictions=[];inner_truth=[];inner_site=[]
                for inner in range(5):
                    if inner==outer:continue
                    val=np.flatnonzero(FOLD23==inner)
                    eligible=[]
                    for forest in S23:
                        iv=val[SITE23[val]==forest]
                        ot=test[SITE23[test]==forest]
                        if len(ot) and len(iv):iv=iv[cKDTree(XY23[ot]).query(XY23[iv],k=1)[0]>=100.]
                        eligible.extend(iv.tolist())
                    val=np.array(sorted(eligible),dtype=int)
                    if not len(val):continue
                    itr,_=training23([outer,inner],val,inner)
                    h=HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
                    h.fit(M21.iloc[itr][features].to_numpy(np.float32),Y23[itr])
                    inner_predictions.append(np.clip(h.predict(
                        M21.iloc[val][features].to_numpy(np.float32)),0,1))
                    inner_truth.append(Y23[val]);inner_site.append(SITE23[val])
                px=np.concatenate(inner_predictions);true=np.concatenate(inner_truth)
                sites=np.concatenate(inner_site)
                cal=IsotonicRegression(y_min=0.,y_max=1.,out_of_bounds='clip')
                cal.fit(px,true,sample_weight=balanced_weights23(true,sites))
                return np.clip(cal.predict(base),0,1)
            P23.loc[test,fs+'__nested_isotonic']=cached23(
                RUN23/f'{fs}_nested_isotonic_outer{outer}.npz',test,train_nested_cal23)
            print('DONE',outer,fs,'nested_isotonic',flush=True)
    pd.DataFrame(audit23).to_csv(RUN23/'23_training_audit.csv',index=False)
    names23=[fs+'__'+method for fs in SETS23 for method in STRATEGIES23]
    assert P23[names23].notna().all().all()
    P23.to_parquet(RUN23/'23_outer_fold_predictions.parquet',index=False)
    MET23=summaries21(P23,names23,'spatial_block_50m')
    MET23['feature_set']=MET23.model.str.split('__').str[0]
    MET23['strategy']=MET23.model.str.split('__').str[1]
    MET23.to_csv(RUN23/'23_scores_by_forest.csv',index=False)
    print('Forest-equal full-heldout performance:')
    display(MET23[MET23.site.eq('forest_equal')][
        ['feature_set','strategy','R2','MAE_pp','RMSE_pp','bias_pp']].round(3))
    print('MOS high-cover diagnostics:')
    display(MET23[MET23.site.eq('MOS')][
        ['feature_set','strategy','R2','MAE_pp','high_bias_pp','high_recall']].round(3))
    # Paired block bootstrap for change in MAE. Do not treat post hoc best
    # method as independently validated; these are exploratory comparisons.
    delta_rows23=[]
    for fs in SETS23:
        base=np.abs(P23[fs+'__baseline'].to_numpy(float)-Y23)*100
        for strategy in STRATEGIES23[1:]:
            delta=100*np.abs(P23[fs+'__'+strategy].to_numpy(float)-Y23)-base
            point=np.mean([delta[SITE23==s].mean() for s in S23])
            rng=np.random.default_rng(SEED23+sum(map(ord,fs+strategy)))
            pools={}
            for forest in S23:
                idx=np.flatnonzero(SITE23==forest)
                pools[forest]=[idx[v] for v in P23.iloc[idx].groupby('block21').indices.values()]
            bs=[]
            for _ in range(300):
                m=[]
                for forest in S23:
                    groups=pools[forest]
                    draws=rng.integers(0,len(groups),size=len(groups))
                    ids=np.concatenate([groups[j] for j in draws])
                    m.append(delta[ids].mean())
                bs.append(np.mean(m))
            low,high=np.quantile(bs,[.025,.975])
            delta_rows23.append(dict(features=fs,strategy=strategy,
                                     delta_MAE_pp=point,ci_low=low,ci_high=high))
    DIFF23=pd.DataFrame(delta_rows23)
    DIFF23.to_csv(RUN23/'23_paired_block_MAE_differences.csv',index=False)
    display(DIFF23.round(3))
    CAL23=[]
    for fs in SETS23:
        for method in STRATEGIES23:
            g=P23[P23.site.eq('MOS')].copy()
            g['cover_class']=pd.cut(g.target_fraction,[-.001,.05,.20,.40,.60,1.001],
                        labels=['0-5','5-20','20-40','40-60','60-100'],include_lowest=True)
            for label,sub in g.groupby('cover_class',observed=True):
                CAL23.append(dict(features=fs,strategy=method,cover_class=str(label),
                                  n=len(sub),observed_pp=100*sub.target_fraction.mean(),
                                  predicted_pp=100*sub[fs+'__'+method].mean(),
                                  bias_pp=100*(sub[fs+'__'+method]-sub.target_fraction).mean()))
    CAL23=pd.DataFrame(CAL23)
    CAL23.to_csv(RUN23/'23_MOS_cover_bin_calibration.csv',index=False)
    for fs in SETS23:
        fig,ax=plt.subplots(figsize=(8.5,6))
        for method in STRATEGIES23:
            g=CAL23[(CAL23.features==fs)&(CAL23.strategy==method)]
            ax.plot(g.observed_pp,g.predicted_pp,'o-',label=method)
        ax.plot([0,100],[0,100],'k:',label='1:1')
        ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed MOS V5 cover (%)',
               ylabel='Predicted cover (%)',title='MOS — '+fs+' spatial-block calibration')
        ax.grid(alpha=.2);ax.legend(fontsize=8)
        fig.savefig(RUN23/('23_MOS_calibration_'+fs+'.png'),dpi=220,bbox_inches='tight')
        plt.show();plt.close(fig)
        fig,ax=plt.subplots(figsize=(7,6))
        for forest in S23:
            g=P23[P23.site.eq(forest)].sample(n=min(175,(SITE23==forest).sum()),
                             random_state=SEED23+sum(map(ord,forest)))
            ax.scatter(100*g.target_fraction,100*g[fs+'__moderate'],
                       s=13,alpha=.5,label=forest)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed V5 cover (%)',
               ylabel='Predicted cover (%)',title=fs+' — moderate weighting')
        ax.legend();ax.grid(alpha=.2)
        fig.savefig(RUN23/('23_balanced_scatter_'+fs+'.png'),dpi=220,bbox_inches='tight')
        plt.show();plt.close(fig)
    print('Section 23 outputs:',RUN23)
    print('Exploratory comparison: model selection needs new independent spatial validation.')
else:print("Abundance sensitivity disabled; baseline retained.",flush=True)


## 10 — Optional independent forest-condition data (ETH 2020, JRC December 2020, Copernicus 2019)
Sources: https://langnico.github.io/globalcanopyheight/ ; https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee ; https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_Landcover_100m_Proba-V-C3_Global . Controls are modelled proxies with temporal mismatch. Coarse source sampling is approximate; do not interpret as 10m measurements.

In [ ]:
# Optional independent controls, freshly downloaded; no historical table.
# Native 30m TMF and 100m tree cover are sampled on the 10m lattice, not new 10m observations.
STRUCT_CFG=dict(version='v5_structure_2020_v1',height='users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1',height_sd='users/nlang/ETH_GlobalCanopyHeightSD_2020_10m_v1',tmf='projects/JRC/TMF/v1_2025/AnnualChanges',tmf_band='Dec2020',classes=[1,2,4],tree='COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019',tree_band='tree-coverfraction',aggregation='50m cell means and height SD; coarse source nearest sampling at 10m',grid=SAT_ID)
STRUCT_ID=digest(STRUCT_CFG)[:20];STRUCT_DIR=ROOT/'structure_10m'/STRUCT_ID;STRUCT_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(STRUCT_DIR/'config.json',STRUCT_CFG)
CONTROLS14=['height_local_mean_m','height_local_sd_m','tmf_undisturbed_local_frac','tmf_degraded_local_frac','tmf_regrowth_local_frac','cop_2019_tree_fraction_pct']
STRUCT_COLS12=CONTROLS14+['height_model_sd_local_m']
BANDS14=S2_BANDS;spec14=dict(zip(BANDS14,ANNUAL[:10]))
STRUCT_STATUS='disabled';D14=pd.DataFrame()
if RUN_STRUCTURE:
 try:
  st_parts=[]
  for (forest,tx,ty),d in D.groupby(['site','tile_x','tile_y'],sort=True):
   tx=int(tx);ty=int(ty)
   h=ee.Image(STRUCT_CFG['height']).select([0]).rename('height')
   sd=ee.Image(STRUCT_CFG['height_sd']).select([0]).rename('height_model_sd')
   tmf=ee.ImageCollection(STRUCT_CFG['tmf']).mosaic().select(STRUCT_CFG['tmf_band']).rename('tmf')
   tree=ee.Image(STRUCT_CFG['tree']).select(STRUCT_CFG['tree_band']).rename('tree')
   bands=['height','height_model_sd','tmf','tree']
   a=fetch_image(ee.Image.cat([h,sd,tmf,tree]).unmask(-9999,sameFootprint=False).toFloat(),bands,STRUCT_DIR,forest,tx,ty,'STRUCT',STRUCT_ID)
   b=tile_bounds(tx,ty);rr=np.floor((b[3]-d.y.to_numpy())/50).astype(int);cc=np.floor((d.x.to_numpy()-b[0])/50).astype(int)
   good=(a!=-9999).all(axis=0)&np.isfinite(a).all(axis=0)&(a[0]>=0)&(a[1]>=0)&np.isin(a[2],[1,2,3,4,5,6])&(a[3]>=0)&(a[3]<=100)
   count=reduce_grid(good.astype(np.int32),50,10)
   def mean(z):
    total=reduce_grid(np.where(good,z,0.),50,10)
    return np.divide(total,count,out=np.full(total.shape,np.nan,dtype=np.float32),where=count>0)
   mh=mean(a[0]);height_sd=np.sqrt(np.maximum(mean(a[0]**2)-mh**2,0))
   fields=[mh,height_sd,mean(a[2]==1),mean(a[2]==2),mean(a[2]==4),mean(a[3]),mean(a[1])]
   z=d.copy().reset_index(drop=True)
   for name,v in zip(STRUCT_COLS12,fields):z[name]=v[rr,cc]
   z['structure_valid_fraction']=count[rr,cc]/25.
   st_parts.append(z[z.structure_valid_fraction.ge(.90)])
  D14=pd.concat(st_parts,ignore_index=True).dropna(subset=STRUCT_COLS12)
  atomic_parquet(D14,ROOT/f'v5_structure_50m_{SIGNATURE}_{STRUCT_ID}.parquet')
  STRUCT_STATUS='complete'
  display(D14.groupby('site').size().rename('valid_structure_cells'))
 except Exception as exc:
  STRUCT_STATUS='failed_optional_access_or_extraction'
  print('OPTIONAL STRUCTURE ANALYSIS UNAVAILABLE:',type(exc).__name__,str(exc)[:500],flush=True)
  atomic_json(RUN/'structure_error.json',dict(status=STRUCT_STATUS,error=str(exc)))
  D14=pd.DataFrame()
atomic_json(RUN/'structure_status.json',dict(status=STRUCT_STATUS,recipe=STRUCT_CFG))


In [ ]:
KEYS11=KEYS
# Matched LOFO fits, deterministic per-forest stratified draw; saved individually.
def draw11(d,forest,cap=1500,seed=20261120):
 rng=np.random.default_rng(seed+sum(map(ord,forest)))
 ids=np.flatnonzero(d.site.to_numpy()==forest)
 if len(ids)<=cap:return ids
 values=d.target_fraction.to_numpy(float)
 strata=np.digitize(values[ids],[.1,.2,.4])
 bins={k:ids[strata==k] for k in range(4)}
 chosen=[]
 # Reserve up to 25% for high-cover cells, if available.
 q=min(len(bins[3]),int(np.ceil(cap*.25)))
 if q:
  t=rng.choice(bins[3],q,replace=False);chosen.extend(t.tolist());bins[3]=np.setdiff1d(bins[3],t)
 while len(chosen)<cap:
  active=[k for k in bins if len(bins[k])]
  assert active
  for k in active:
   nn=min(len(bins[k]),max(1,(cap-len(chosen))//len(active)),cap-len(chosen))
   if nn:
    t=rng.choice(bins[k],nn,replace=False)
    chosen.extend(t.tolist());bins[k]=np.setdiff1d(bins[k],t)
   if len(chosen)==cap:break
 return np.asarray(sorted(chosen),int)
def fit_matched_lofo(data,featuresets,out,seed=20261120):
 yv=data.target_fraction.to_numpy(float)
 forestv=data.site.to_numpy()
 per=[];preds=[]
 for held in SITES:
  test=np.flatnonzero(forestv==held)
  train_ids=np.concatenate([draw11(data,s,seed=seed) for s in SITES if s!=held])
  assert len(np.intersect1d(test,train_ids))==0
  frame=data.iloc[test][KEYS11+['target_fraction']].reset_index(drop=True).copy()
  for name,cols in featuresets.items():
   file=out/f'{held}_{name}.npz'
   if file.is_file():
    with np.load(file,allow_pickle=False) as z:
     assert np.array_equal(z['test'],test)
     yp=z['pred'].astype(float)
    print('CACHE',held,name,flush=True)
   else:
    h=HistGradientBoostingRegressor(random_state=seed,**MODEL)
    h.fit(data.iloc[train_ids][cols].to_numpy(np.float32),yv[train_ids])
    yp=np.clip(h.predict(data.iloc[test][cols].to_numpy(np.float32)),0,1)
    tmp=file.with_suffix('.partial')
    with open(tmp,'wb') as fd:np.savez_compressed(fd,test=test,pred=yp.astype(np.float32))
    os.replace(tmp,file)
    print('FIT',held,name,flush=True)
   frame[name]=yp
   err=yp-yv[test];variance=np.var(yv[test])
   hi=yv[test]>=.4
   per.append(dict(site=held,model=name,n=len(test),n_train=len(train_ids),
     MAE_pp=100*np.mean(abs(err)),RMSE_pp=100*np.sqrt(np.mean(err**2)),
     MSE=np.mean(err**2),bias_pp=100*np.mean(err),
     R2=1-np.mean(err**2)/variance if variance>0 else np.nan,
     high_n=int(hi.sum()),high_bias_pp=100*np.mean(err[hi]) if hi.any() else np.nan,
     high_recall=float(np.mean(yp[hi]>=.4)) if hi.any() else np.nan))
  preds.append(frame)
 PRED=pd.concat(preds,ignore_index=True)
 PER=pd.DataFrame(per)
 mu=PRED.groupby('site').target_fraction.mean().mean()
 variance=np.mean([np.mean((g.target_fraction.to_numpy()-mu)**2) for _,g in PRED.groupby('site')])
 SUM=[]
 for name in featuresets:
  g=PER[PER.model.eq(name)]
  SUM.append(dict(model=name,n=len(PRED),n_forests=4,
   equal_forest_R2=1-g.MSE.mean()/variance,
   equal_forest_MAE_pp=g.MAE_pp.mean(),
   equal_forest_RMSE_pp=np.sqrt(np.mean(g.RMSE_pp**2)),
   MOS_R2=float(g[g.site.eq('MOS')].R2.iloc[0]),
   MOS_high_bias_pp=float(g[g.site.eq('MOS')].high_bias_pp.iloc[0]),
   MOS_high_recall=float(g[g.site.eq('MOS')].high_recall.iloc[0])))
 SUM=pd.DataFrame(SUM)
 PER.to_csv(out/'LOFO_per_forest.csv',index=False)
 SUM.to_csv(out/'LOFO_equal_forest_summary.csv',index=False)
 PRED.to_parquet(out/'LOFO_matched_predictions.parquet',index=False)
 return PRED,PER,SUM

if STRUCT_STATUS=='complete' and set(D14.site)==set(SITES) and D14.groupby('site').size().min()>=30:
    PSETS12={'forest_conditions_only':STRUCT_COLS12,'S2_annual_only':ANNUAL[:14],'forest_conditions_plus_S2':STRUCT_COLS12+ANNUAL[:14]}
    RUN12=OUT/'forest_condition_controls'/digest(dict(source=SOURCE_SHA,structure=STRUCT_ID,table=dataframe_sha(D14[KEYS+STRUCT_COLS12]),model=MODEL))[:20];RUN12.mkdir(parents=True,exist_ok=True)
    PRED12,PER12,SUM12=fit_matched_lofo(D14,PSETS12,RUN12)
    display(SUM12.round(3))
else:print('Insufficient independent control coverage for four-forest LOFO.')


## 11 — Balance-gated matched high/low spectral contrasts
For tractability, matching uses an outcome-blind deterministic random cap of 2,500 complete cells per forest. All six controls must balance; no species-specific attribution. No matching on S2 spectra.

In [ ]:
if STRUCT_STATUS=='complete' and len(D14)>0:
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial.distance import cdist
    SEED17=20261120
    SPECS17={'strict':dict(height=5.,tree=10.,regrowth=.20,dist=2.0),
             'sensitivity':dict(height=8.,tree=15.,regrowth=.30,dist=3.0)}
    CONT17=CONTROLS14
    OUT17=RUN/'balance_qualified_matching'
    OUT17.mkdir(parents=True,exist_ok=True)
    MIN_PAIRS17=15;MIN_BLOCKS17=5;SMD_LIMIT17=.10
    def match17(forest,spec):
     d=D14[D14.site.eq(forest)].reset_index(drop=True)
     if len(d)>2500:d=d.sample(n=2500,random_state=SEED17).reset_index(drop=True)
     h=d[d.target_fraction>=.40].reset_index(drop=True)
     l=d[d.target_fraction<=.05].reset_index(drop=True)
     info=dict(site=forest,specification=spec,n_high=len(h),n_low=len(l),n_initial=0,n_retained=0,
               n_blocks=0,max_abs_smd=np.nan,accepted=False,reason='')
     if len(h)==0 or len(l)==0:
      info['reason']='No high/low common abundance classes';return [],info,[]
     source=d[CONT17].to_numpy(float);sd=source.std(axis=0,ddof=1)
     active=sd>1e-10;sd=np.where(active,sd,1.)
     if not active.all():
      info['reason']='One or more constant control covariates; overlap not established';return [],info,[]
     zh=h[CONT17].to_numpy(float)/sd;zl=l[CONT17].to_numpy(float)/sd
     dist=cdist(zh,zl)
     rule=SPECS17[spec]
     spatial=cdist(h[['x','y']],l[['x','y']])
     eligible=(np.abs(h.height_local_mean_m.to_numpy()[:,None]-l.height_local_mean_m.to_numpy()[None,:])<=rule['height'])
     eligible&=(np.abs(h.cop_2019_tree_fraction_pct.to_numpy()[:,None]-l.cop_2019_tree_fraction_pct.to_numpy()[None,:])<=rule['tree'])
     eligible&=(np.abs(h.tmf_regrowth_local_frac.to_numpy()[:,None]-l.tmf_regrowth_local_frac.to_numpy()[None,:])<=rule['regrowth'])
     eligible&=(dist<=rule['dist'])&(spatial>=150.)
     # Dummy columns allow high cells to remain unmatched.
     penalty=10000.
     cost=np.concatenate((np.where(eligible,dist,penalty*2),np.full((len(h),len(h)),penalty)),axis=1)
     ii,jj=linear_sum_assignment(cost)
     pairs=[(int(i),int(j)) for i,j in zip(ii,jj) if j<len(l) and eligible[i,j]]
     info['n_initial']=len(pairs)
     # Outcome-blind balance trimming: remove the pair most contributing to the most imbalanced mean.
     # Never use spectral measurements, and stop before the minimum required sample size.
     while len(pairs)>MIN_PAIRS17:
      differences=np.array([zh[i]-zl[j] for i,j in pairs])
      smd=differences.mean(axis=0)
      if np.max(np.abs(smd))<=SMD_LIMIT17:break
      worst=int(np.argmax(np.abs(smd)))
      contributions=np.sign(smd[worst])*differences[:,worst]
      pairs.pop(int(np.argmax(contributions)))
     if pairs:
      diffs=np.array([zh[i]-zl[j] for i,j in pairs])
      smd=diffs.mean(axis=0)
      info['max_abs_smd']=float(np.max(np.abs(smd)))
     else:smd=np.full(len(CONT17),np.nan)
     rows=[];blocks=set()
     for ix,(i,j) in enumerate(pairs):
      a=h.iloc[i];b=l.iloc[j]
      block=f"{forest}_{int(a.x//500)}_{int(a.y//500)}";blocks.add(block)
      record=dict(site=forest,specification=spec,pair=ix,block500=block,
                  high_cover=float(a.target_fraction),low_cover=float(b.target_fraction),
                  high_x=float(a.x),high_y=float(a.y),low_x=float(b.x),low_y=float(b.y))
      for feature in CONT17:
       record['high_'+feature]=float(a[feature]);record['low_'+feature]=float(b[feature])
      for band,col in spec14.items():record['delta_'+band]=float(a[col]-b[col])
      rows.append(record)
     info['n_retained']=len(rows);info['n_blocks']=len(blocks)
     info['accepted']=bool(len(rows)>=MIN_PAIRS17 and len(blocks)>=MIN_BLOCKS17 and np.all(np.abs(smd)<=SMD_LIMIT17))
     info['reason']='Balance and support pass' if info['accepted'] else 'Failed balance and/or support; spectral attribution inconclusive'
     balance=[dict(site=forest,specification=spec,control=feature,smd=float(v),passes=bool(abs(v)<=SMD_LIMIT17)) for feature,v in zip(CONT17,smd)] if rows else []
     return rows,info,balance
    rows17=[];audit17=[];bal17=[]
    for setting in SPECS17:
     for forest in SITES:
      r,a,b=match17(forest,setting);rows17.extend(r);audit17.append(a);bal17.extend(b)
    PAIRS17=pd.DataFrame(rows17);AUDIT17=pd.DataFrame(audit17);BAL17=pd.DataFrame(bal17)
    AUDIT17.to_csv(OUT17/'17_matching_attrition.csv',index=False)
    BAL17.to_csv(OUT17/'17_covariate_balance.csv',index=False)
    if len(PAIRS17):PAIRS17.to_parquet(OUT17/'17_matched_pairs.parquet',index=False)
    display(AUDIT17.round(3))
    if len(BAL17):display(BAL17.pivot_table(index=['site','specification'],columns='control',values='smd').round(3))
    
    def ci18(df,col,reps=2000,seed=20261120):
     groups=[g[col].to_numpy(float) for _,g in df.groupby('block500')]
     if len(groups)<MIN_BLOCKS17:return np.nan,np.nan
     sums=np.array([g.sum() for g in groups]);counts=np.array([len(g) for g in groups])
     rng=np.random.default_rng(seed);take=rng.integers(0,len(groups),(reps,len(groups)))
     dist=sums[take].sum(axis=1)/counts[take].sum(axis=1)
     return np.quantile(dist,[.025,.975])
    spectral18=[]
    for item in AUDIT17.itertuples():
     if not item.accepted:continue
     f=PAIRS17[(PAIRS17.site==item.site)&(PAIRS17.specification==item.specification)]
     for band in BANDS14:
      col='delta_'+band;low,high=ci18(f,col,seed=SEED17+sum(map(ord,item.site))+BANDS14.index(band))
      spectral18.append(dict(site=item.site,specification=item.specification,band=band,n_pairs=len(f),
          n_blocks=f.block500.nunique(),difference=float(f[col].mean()),ci_low=low,ci_high=high))
    SPECTRAL18=pd.DataFrame(spectral18,columns=['site','specification','band','n_pairs','n_blocks','difference','ci_low','ci_high'])
    SPECTRAL18.to_csv(OUT17/'18_balance_qualified_spectral.csv',index=False)
    if SPECTRAL18.empty:print('NO BALANCE-QUALIFIED SPECTRAL CONTRAST. NB08 previous B8 association is not confirmed after adequate balancing.')
    else:display(SPECTRAL18[SPECTRAL18.band.eq('B8')].round(5))
else:print("Independent controls missing; matched spectra not estimated.",flush=True)


## 12 — MOS continuous conditional spectra and high-cover slope change
No extrapolation outside observed 1st–99th percentile cover range; skip if support is insufficient.

In [ ]:
mos=D14[D14.site.eq('MOS')] if len(D14) else pd.DataFrame()
ready22=(len(mos)>=100 and (mos.target_fraction>=.4).sum()>=30 and mos.target_fraction.quantile(.01)<=.05 and mos.target_fraction.quantile(.99)>.65 and len(mos.groupby([mos.x.floordiv(500),mos.y.floordiv(500)]))>=15) if len(mos) else False
if ready22:
    # SECTION 22 — Continuous structure-controlled spectral response at MOS
    # Adjusted NIR/red-edge/SWIR curves; spatial-block bootstrap for conditional association.
    from sklearn.preprocessing import SplineTransformer,StandardScaler
    from sklearn.linear_model import Ridge
    from scipy.stats import spearmanr
    assert 'D14' in globals() and 'CONTROLS14' in globals() and 'spec14' in globals(), 'Run native structure extraction first'
    assert 'D4' in globals() and D.equals(D4), 'This must use the V5 NB07 binary-cover reference'
    SITE22='MOS'
    SEED22=20261123
    BOOT22=300
    BLOCK22=500.
    ALPHA22=10.
    KNOTS22=np.array([0.,.10,.20,.40,.60,.80,1.],dtype=float).reshape(-1,1)
    COL22=['B8','B5','B6','B7','B8A','B11','B12']
    COV22=list(CONTROLS14)
    M22=D14[D14.site.eq(SITE22)].copy().replace([np.inf,-np.inf],np.nan)
    M22=M22.dropna(subset=['target_fraction','x','y']+COV22+[spec14[b] for b in COL22]).reset_index(drop=True)
    assert len(M22)>=100 and M22.target_fraction.between(0,1).all()
    assert (M22.target_fraction>=.4).sum()>=30, 'Not enough high-cover MOS cells for continuous response'
    M22['block22']=np.floor(M22.x/BLOCK22).astype('int64').astype(str)+'_'+np.floor(M22.y/BLOCK22).astype('int64').astype(str)
    BLOCKS22=list(M22.groupby('block22').indices.values())
    assert len(BLOCKS22)>=15,('Too few independent spatial blocks',len(BLOCKS22))
    sha22=hashlib.sha256(pd.util.hash_pandas_object(
        M22[['site','mosaic','row','col','target_fraction']+COV22+[spec14[b] for b in COL22]],
        index=False).values.tobytes()).hexdigest()
    CFG22=dict(version='V5_MOS_continuous_conditional_spectral_v1',
        source_sha=SOURCE_SHA,matched_sample_sha=sha22,
        block_m=BLOCK22,bootstraps=BOOT22,ridge_alpha=ALPHA22,
        spline_knots=KNOTS22.ravel().tolist(),covariates=COV22,bands=COL22,
        inferential_scope='conditional association; not species-specific causal attribution')
    RUN22=OUT/'26_NB08_MOS_continuous_spectral_response'/hashlib.sha256(
        json.dumps(CFG22,sort_keys=True).encode()).hexdigest()[:14]
    RUN22.mkdir(parents=True,exist_ok=True)
    config22=RUN22/'22_config.json'
    if config22.exists():assert json.loads(config22.read_text())==CFG22
    else:config22.write_text(json.dumps(CFG22,indent=2))
    # Fixed spline basis and standardization across block-bootstrap replicates.
    # Predict at the same measured covariate mean. This additive model does not
    # impose a monotonic response; flattening must be supported by intervals.
    spline22=SplineTransformer(knots=KNOTS22,degree=2,include_bias=False,
                               extrapolation='constant')
    ycover22=M22.target_fraction.to_numpy(float)
    sp22=spline22.fit_transform(ycover22.reshape(-1,1))
    standard22=StandardScaler()
    xcov22=standard22.fit_transform(M22[COV22].to_numpy(float))
    X22=np.column_stack([sp22,xcov22])
    Y22=M22[[spec14[b] for b in COL22]].to_numpy(float)
    assert np.isfinite(X22).all() and np.isfinite(Y22).all()
    low22=max(0.,float(np.quantile(ycover22,.01)))
    high22=min(1.,float(np.quantile(ycover22,.99)))
    assert high22>.65,(low22,high22,'Insufficient high-cover support')
    grid22=np.linspace(low22,high22,81)
    Xgrid22=np.column_stack([
        spline22.transform(grid22.reshape(-1,1)),
        np.zeros((len(grid22),len(COV22)),dtype=float)])
    def fit22(indices):
        reg=Ridge(alpha=ALPHA22,fit_intercept=True)
        reg.fit(X22[indices],Y22[indices])
        return reg.predict(Xgrid22)
    point22=fit22(np.arange(len(M22)))
    rng22=np.random.default_rng(SEED22)
    boot22=np.empty((BOOT22,len(grid22),len(COL22)),dtype=np.float32)
    for rep in range(BOOT22):
        draw=rng22.integers(0,len(BLOCKS22),size=len(BLOCKS22))
        ids=np.concatenate([BLOCKS22[j] for j in draw])
        boot22[rep]=fit22(ids)
        if (rep+1)%100==0:print('MOS spectral block bootstrap',rep+1,'/',BOOT22,flush=True)
    l22=np.quantile(boot22,.025,axis=0)
    u22=np.quantile(boot22,.975,axis=0)
    rows22=[]
    for k,band in enumerate(COL22):
        for t,value in enumerate(grid22):
            rows22.append(dict(band=band,cover_fraction=value,cover_pct=100*value,
                adjusted_reflectance=point22[t,k],ci_low=l22[t,k],ci_high=u22[t,k]))
    CURVE22=pd.DataFrame(rows22)
    CURVE22.to_csv(RUN22/'22_MOS_adjusted_spectral_response.csv',index=False)
    # Observed-bin diagnostics, not treated as fitted adjusted effects.
    bins22=[-.001,.05,.20,.40,.60,1.001]
    labels22=['0-5','5-20','20-40','40-60','60-100']
    M22['abundance_bin22']=pd.cut(ycover22,bins22,labels=labels22,include_lowest=True)
    BIN22=M22.groupby('abundance_bin22',observed=True).agg(
        n=('target_fraction','size'),mean_cover=('target_fraction','mean'),
        mean_B8=(spec14['B8'],'mean'),sd_B8=(spec14['B8'],'std')).reset_index()
    BIN22.to_csv(RUN22/'22_MOS_observed_abundance_bins.csv',index=False)
    display(BIN22)
    # Contrasts from the same regression curve; uncertainty uses block bootstrap.
    def contrast22(a,b,band='B8'):
        assert low22<=a<=high22 and low22<=b<=high22,(a,b,low22,high22)
        j=COL22.index(band)
        # Interpolate every bootstrap curve on identical cover grid.
        delta=float(np.interp(b,grid22,point22[:,j])-np.interp(a,grid22,point22[:,j]))
        empirical=np.array([np.interp(b,grid22,v[:,j])-np.interp(a,grid22,v[:,j])
                            for v in boot22],dtype=float)
        lo,hi=np.quantile(empirical,[.025,.975])
        return dict(band=band,from_pct=a*100,to_pct=b*100,
                    adjusted_difference=delta,ci_low=lo,ci_high=hi,
                    n=len(M22),n_blocks=len(BLOCKS22))
    CONTRAST22=pd.DataFrame([contrast22(a,b,band)
        for band in COL22 for a,b in [(.05,.40),(.40,.65)]])
    CONTRAST22.to_csv(RUN22/'22_MOS_continuous_response_contrasts.csv',index=False)
    display(CONTRAST22[CONTRAST22.band.eq('B8')].round(5))
    fig,ax=plt.subplots(figsize=(9,6))
    ix=COL22.index('B8')
    ax.plot(grid22*100,point22[:,ix],color='#185e78',label='Structure-adjusted B8')
    ax.fill_between(grid22*100,l22[:,ix],u22[:,ix],alpha=.22,color='#185e78',
                    label='500 m block bootstrap 95% CI')
    ax.scatter(BIN22.mean_cover*100,BIN22.mean_B8,s=BIN22.n.clip(5,100),
               color='#bc762b',edgecolor='black',zorder=4,label='Observed cover-bin means')
    ax.set(xlabel='UAV V5 Musanga cover (%)',ylabel='Sentinel-2 B8 reflectance',
           title='MOS — conditional continuous B8 spectral response')
    ax.grid(alpha=.2);ax.legend()
    fig.savefig(RUN22/'22_MOS_B8_continuous_adjusted_response.png',dpi=240,bbox_inches='tight')
    plt.show();plt.close(fig)
    fig,axs=plt.subplots(2,3,figsize=(14,8),layout='constrained')
    for ax,band in zip(axs.flat,[b for b in COL22 if b!='B8']):
        k=COL22.index(band)
        ax.plot(grid22*100,point22[:,k],color='#185e78')
        ax.fill_between(grid22*100,l22[:,k],u22[:,k],alpha=.2,color='#185e78')
        ax.set(xlabel='Musanga cover (%)',ylabel='Reflectance',title=band)
        ax.grid(alpha=.15)
    fig.savefig(RUN22/'22_MOS_rededge_SWIR_conditional_response.png',
                dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
    QA22=dict(n_cells=int(len(M22)),n_blocks=len(BLOCKS22),
              high_cover_cells=int((ycover22>=.4).sum()),
              cover_min=float(ycover22.min()),cover_max=float(ycover22.max()),
              range_plotted=[float(low22),float(high22)],forest=SITE22,
              target='NB07 V5 band 2 fraction',
              caveat='Conditional additive association; controls may be downstream of Musanga; uncertainty does not include UAV segmentation/reference error.')
    (RUN22/'22_interpretation_and_QA.json').write_text(json.dumps(QA22,indent=2))
    print('Section 22 complete:',RUN22)
    print('Interpretation: association conditional on measured structure and disturbance; not causal attribution.')
    
    # SECTION 24 — explicit slope-change test from Section 22 spatial-block replicates
    assert 'boot22' in globals() and 'point22' in globals() and 'grid22' in globals(), 'Run Section 22 first'
    assert 'COL22' in globals() and 'RUN22' in globals() and 'RUN22' in globals()
    SLOPE24=[]
    for band in COL22:
        k=COL22.index(band)
        def slopes(curve):
            a=np.interp(.05,grid22,curve[:,k])
            b=np.interp(.40,grid22,curve[:,k])
            c=np.interp(.65,grid22,curve[:,k])
            lower=(b-a)/.35
            upper=(c-b)/.25
            return lower,upper,upper-lower
        vals=np.array([slopes(curve) for curve in boot22])
        estimate=slopes(point22)
        ci=np.quantile(vals[:,2],[.025,.975])
        SLOPE24.append(dict(band=band,n_blocks=len(BLOCKS22),
            low_slope_per_fraction=estimate[0],
            high_slope_per_fraction=estimate[1],
            high_minus_low_slope=estimate[2],
            ci_low=ci[0],ci_high=ci[1],
            bootstrap_proportion_negative=float(np.mean(vals[:,2]<0))))
    SLOPE24=pd.DataFrame(SLOPE24)
    SLOPE24.to_csv(RUN22/'24_MOS_spatial_bootstrap_slope_change.csv',index=False)
    display(SLOPE24.round(5))
    fig,ax=plt.subplots(figsize=(8,4.7))
    rows=SLOPE24.set_index('band').loc[COL22]
    y=np.arange(len(rows))
    ax.errorbar(rows.high_minus_low_slope,y,
        xerr=np.vstack([rows.high_minus_low_slope-rows.ci_low,
                        rows.ci_high-rows.high_minus_low_slope]),
        fmt='o',capsize=3)
    ax.axvline(0,color='black',linestyle=':')
    ax.set(yticks=y,yticklabels=rows.index,
           xlabel='Adjusted slope difference: high (40–65%) minus low (5–40%)',
           title='MOS — block bootstrap test of spectral response flattening')
    ax.grid(axis='x',alpha=.2)
    fig.savefig(RUN22/'24_MOS_spectral_slope_change.png',dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
    print('Section 24 complete: changes relative to Section 22 estimates, not causal effects.')
else:print("Insufficient complete MOS structure/abundance/block support; no conditional slope claim.",flush=True)


## 13 — Fit frozen 50m S2+embedding baseline and save final completion manifest

In [ ]:
# Prespecified primary deployment baseline (not selected by optimistic V5 CV tuning).
import joblib
from sklearn import __version__ as sklearn_version
features=list(F21['M3_S2_EMB'])
train=[]
for forest in SITES:
 pool=np.flatnonzero(M21.site.to_numpy()==forest)
 rng=np.random.default_rng(SEED21+sum(map(ord,forest)))
 if len(pool)>CAP21:
  strata=np.digitize(M21.iloc[pool].target_fraction.to_numpy(),[.10,.20,.40]);take=[]
  for k in range(4):
   ids=pool[strata==k]
   if len(ids):take.extend(rng.choice(ids,min(len(ids),CAP21//4),replace=False).tolist())
  rest=np.setdiff1d(pool,np.array(take,dtype=int))
  take.extend(rng.choice(rest,CAP21-len(take),replace=False).tolist());pool=np.array(sorted(take),int)
 train.extend(pool.tolist())
 train=np.asarray(train,dtype=int).tolist()
prod=M21.iloc[train].copy()
assert set(prod.site)==set(SITES)
model=HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
model.fit(prod[features].to_numpy(np.float32),prod.target_fraction.to_numpy(np.float32))
PROD=OUT/'production_50m'/SIGNATURE;PROD.mkdir(parents=True,exist_ok=True)
tmp=PROD/'model.joblib.partial';joblib.dump(model,tmp);os.replace(tmp,PROD/'model.joblib')
atomic_parquet(prod[KEYS+['x','y','target_fraction']+features],PROD/'training_cells.parquet')
spec=dict(version='V5',status='fitted_teacher_model_not_external_validation',support_m=50,model='HistGradientBoostingRegressor',model_file='model.joblib',sklearn_version=sklearn_version,features=features,model_parameters=MODEL,source_sha=SOURCE_SHA,target_id=TARGET_ID,satellite_id=SAT_ID,training_cap_per_forest=CAP21,training_seed=SEED21,n_train=len(prod),counts=prod.site.value_counts().to_dict(),training_cover_strata=[.10,.20,.40],strategy='pre-specified S2+EMB baseline, no extra abundance weights or calibration',grid_crs=CRS,grid_origin=[0,0],target='NB07 binary band 2 on 1m sampling lattice',selection_note='Prespecified from exploratory V4 scientific analysis; evaluate V5 contrasts before changing production strategy.',spatial_cv_metrics=json.loads(MET21[(MET21.model=='M3_S2_EMB')&(MET21.validation=='spatial_block')].to_json(orient='records')),lofo_metrics=json.loads(MET21[(MET21.model=='M3_S2_EMB')&(MET21.validation=='LOFO')].to_json(orient='records')))
atomic_json(PROD/'25_production_model_spec.json',spec)
# Final handoff is explicit about optional archived experiments that were not regenerated.
complete=dict(schema='musanga_v5_nb08_native',primary_complete=True,reference=str(REF),target_config=str(TARGET_DIR/'config.json'),satellite_config=str(SAT_DIR/'config.json'),n_50m=len(D),counts=D.site.value_counts().to_dict(),source_sha=SOURCE_SHA,scale_run=str(SCALE_RUN),lofo=str(RUN),spatial_cv=str(RUN21),production=str(PROD),structure_status=STRUCT_STATUS,native_area_QA=str(AREA_QA_PATH),abundance_sensitivity='complete' if RUN_ABUNDANCE_SENSITIVITY and 'MET23' in globals() else 'disabled',nicfi_context='not_regenerated; historic experiment preserved in GitHub V4',uav_dsm_context='not_regenerated; historic experiment preserved in GitHub V4',cleanup_authorized=False)
atomic_json(OUT/'08_native_completion_manifest.json',complete)
display(MET21[MET21.model.eq('M3_S2_EMB')].round(3))
print('Native V5 NB08 primary workflow complete:',OUT)
print('Do not remove historical data until ancillary retention decision and independent V5 reproducibility audit.')
